# Actualizar .xlsx de una cartera 

# Estructura del sistema
📦 Sistema de Análisis de Cartera
├── 📄 01_GestorRutas.py
├── 📄 02_GestorCartera.py
├── 📄 03_DescargadorYahoo.py
├── 📄 04_AnalizadorMetricas.py
├── 📄 05_AnalizadorTecnico.py
├── 📄 06_AnalizadorCartera.py
├── 📄 07_Exportador.py
├── 📄 main.py (Orquestador)
└── 📓 Consulta_github_actualizar_cartera.ipynb (Notebook Jupyter)

# Flujo de ejecución
┌─────────────────────────────────┐
│  Iniciar MainAnalisisCartera    │
└────────────┬────────────────────┘
             │
      ┌──────▼──────┐
      │ Menú Principal
      └──────┬──────┘
             │
    ┌────────┴────────┐
    │                 │
    ▼                 ▼
Gestionar         Descargar
Cartera           Datos
    │                 │
    ▼                 ▼
 Análisis          Análisis
Fundamental       Técnico
    │                 │
    └────────┬────────┘
             │
             ▼
      Análisis de
       Cartera
             │
             ▼
         Exportar
        Resultados

# Docstring

"""
╔══════════════════════════════════════════════════════════════════════════════╗
║          GESTOR DE CARTERA DE VALORES — actualizar_cartera.py               ║
║          Versión 2.0  ·  Python 3.10+  ·  Jupyter / Google Colab / CLI      ║
╚══════════════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. QUÉ HACE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Gestor completo de carteras de inversión para el inversor particular.
  Combina tres capas de análisis:

  a) GESTIÓN DE CARTERA
     · Crea y almacena carteras en Excel con estructura estándar.
     · Descarga cotizaciones históricas de Yahoo Finance (fondos, ETFs,
       acciones e índices) directamente en Datos_CSV/.
     · Actualiza automáticamente Valor_actual, Peso%, Benef. y Plusv.%
       usando el precio más reciente del CSV de cada valor.
     · Soporta múltiples aportaciones por valor (suscripciones parciales,
       traspasos, compras adicionales).
     · Compatible con fondos de inversión (NAV diario), ETFs, acciones
       españolas/europeas/USA y planes de pensiones (CSV manual).

  b) ANÁLISIS CUANTITATIVO
     · Métricas básicas: CAGR, volatilidad diaria y mensual, Sharpe,
       Max Drawdown, RSI(14), Beta, Alpha, R².
     · Métricas avanzadas de distribución: Sortino, Omega, Skewness
       (inclinación), Kurtosis (curtosis) y Tail Ratio (relación de cola).
     · Metodología Morningstar: Sharpe y Sortino calculados con retornos
       mensuales y Euribor 3M del periodo como tasa libre de riesgo.
     · Benchmark automático por categoría (BENCHMARK_CATEGORIAS): renta
       fija → iShares Euro Govt Bond 1-3yr; acciones .MC → Ibex 35, etc.
     · Correlación estática y rodante entre fondos; detección automática
       de convergencias (señal de crisis) y pares con alta correlación.
     · Frontera eficiente Markowitz (Monte Carlo 8.000 simulaciones +
       optimización scipy): carteras de Sharpe máximo y mínima varianza.
     · VaR histórico, paramétrico y CVaR (Expected Shortfall) en % y €.
     · Contribución marginal al riesgo por fondo.

  c) VISUALIZACIÓN Y EXPORTACIÓN
     · Gráfica base 100 con benchmark superpuesto (zonas verde/rojo).
     · Gráfica técnica: SMA 20/50/200 + Bollinger + MACD + ATR + RSI.
     · Gráfico subacuático: NAV + drawdown continuo + histograma de
       retornos mensuales con curva normal teórica + tiempo bajo el agua
       por año.
     · Heatmap mensual (año × mes) y comparativo anual (fondo × año).
     · Correlación rodante con bandas de referencia.
     · Exportación PDF (portada + diagnóstico + gráficas) y Excel.

  Por qué combina mercado y análisis estadístico avanzado:
     Los indicadores técnicos (SMA, MACD) detectan señales de corto plazo;
     las métricas de distribución (Skewness, Tail Ratio) revelan si el
     perfil de riesgo real coincide con el declarado; la frontera eficiente
     guía la asignación estratégica. La combinación permite una visión
     completa: ¿está batiendo al benchmark? ¿es el riesgo simétrico?
     ¿está bien diversificada la cartera?


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2. CÓMO SE USA
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ── USO RÁPIDO ──────────────────────────────────────────────────────────────

    # Desde terminal:
    python actualizar_cartera.py

    # Especificando el Excel directamente:
    python actualizar_cartera.py Carteras/mi_cartera_2026.xlsx

    # Desde Jupyter Notebook / Google Colab:
    %run actualizar_cartera.py
    %run actualizar_cartera.py Carteras/mi_cartera_2026.xlsx

    # En Google Colab — variable de entorno para parametrizar sin tocar código:
    import os
    os.environ["CARTERA_EXCEL"] = "/content/drive/MyDrive/Gestor_Cartera/Carteras/mi.xlsx"
    %run actualizar_cartera.py

  ── FLUJO DE PRIMERA EJECUCIÓN ───────────────────────────────────────────────

    1. El programa detecta el entorno (local / Colab) automáticamente.
    2. Busca Excels en Carteras/. Si no hay ninguno, lanza el asistente
       de creación interactivo: nombre → selección de valores del catálogo
       → datos de cada aportación → guarda en Carteras/.
    3. Descarga las cotizaciones de los valores de la cartera → Datos_CSV/.
    4. Actualiza el Excel y muestra el menú principal.

  ── FLUJO SEMANAL HABITUAL ───────────────────────────────────────────────────

    1. Ejecutar el script → seleccionar cartera → responder S a
       "¿Actualizar cotizaciones?" → el Excel queda al día.
    2. Opción 4 → métricas y diagnóstico.
    3. Opción 6 → exportar informe PDF.

  ── MENÚ PRINCIPAL ───────────────────────────────────────────────────────────

    1  Descargar cotizaciones desde internet  (catálogo FONDOS_DATA → Descargas_yahoo/)
    2  Actualizar cartera desde CSVs en disco (Datos_CSV/ → Excel)
    3  Informe completo de un valor           (ficha + métricas + técnico + PDF)
    4  Métricas y diagnóstico                 (todas las métricas por fondo)
    5  Correlaciones y medias móviles         (matriz + rodante + MM)
    6  Exportar informe PDF + Excel           (informe completo de la cartera)
    7  Análisis técnico avanzado              (SMA/Bollinger/MACD/ATR + PDF)
    8  Análisis de cartera agregado           (Markowitz + VaR + RC)
    9  Contextualización                      (benchmark + heatmaps)
    A  Crear nueva cartera                    (asistente interactivo)
    B  Cambiar cartera activa                 (seleccionar de Carteras/)
    0  Salir

  ── ESTRUCTURA DE CARPETAS ───────────────────────────────────────────────────

    Mi_Cartera/
    ├── actualizar_cartera.py
    ├── Carteras/              ← Excels de carteras (*.xlsx)
    ├── Datos_CSV/             ← CSVs curados para análisis (fuente de verdad)
    ├── Descargas_yahoo/       ← Descargas automáticas del catálogo completo
    ├── exports/               ← Informes PDF y Excel exportados
    ├── logs/                  ← Archivo de registro de actualizaciones
    └── Alertas/               ← Reservado para alertas automáticas (futuro)

  ── ACTIVOS SOPORTADOS ───────────────────────────────────────────────────────

    · Fondos de inversión europeos  : ticker 0PXXXXXXXXX.F (Morningstar/Yahoo)
    · ETFs                          : ticker EXSA.DE, URTH, IBGS, etc.
    · Índices                       : ^IBEX, ^GSPC, ^STOXX50E, ^GDAXI, etc.
    · Acciones españolas            : IBE.MC, REP.MC, TEF.MC, etc.
    · Acciones europeas             : ENI.MI, ORA.PA, NDA-SE.ST, etc.
    · Acciones USA                  : AAPL, MSFT, GOOGL, etc.
    · Planes de pensiones           : CSV manual con columnas Date y Close

  ── PERSONALIZAR EL CATÁLOGO ─────────────────────────────────────────────────

    Edita el diccionario FONDOS_DATA al inicio del script:

      FONDOS_DATA = {
          "Mi categoría": {
              "Nombre del fondo": {"ISIN": "IE00XXXXXXXX", "ticker": "0PXXXXXXXX.F"},
          },
      }

    Y el diccionario BENCHMARK_CATEGORIAS para asignar benchmarks:

      BENCHMARK_CATEGORIAS = {
          "IE00XXXXXXXX": "IE00B4WXJJ64",  # → iShares Euro Govt Bond 1-3yr
          ".MC"         : "ESI143420005",  # acciones españolas → Ibex 35
      }


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━��━━━━━━━
3. CÓMO INTERPRETAR LOS RESULTADOS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ── TABLA DE MÉTRICAS BÁSICAS (opción 4) ─────────────────────────────────────

    Columna     Descripción                    Orientativo          Alarma
    ─────────────────────────────────────────────────────────────────────────
    NAV         Precio actual participación    —                    —
    Rend%       Rentabilidad periodo           >Rf del periodo      <0%
    CAGR%/a     Rentabilidad anual compuesta   >2% (RF) >8% (RV)   <0%
    Vol%        Volatilidad anual (mensual×√12) <2% RF  <20% RV    >25%
    Sharpe      (CAGR-Rf)/Vol mensual          >1 bueno            <0
    Sortino     (CAGR-Rf)/Vol_bajista          >1.0 bueno          <0
    MaxDD%      Caída máxima desde máximo      >-5% RF             <-15%
    RSI(14)     Fuerza relativa                30–70 neutral       >80/<20
    Años        Antigüedad del CSV en el rango  >3 para fiabilidad  <1
    Alpha%/a    Exceso de rentabilidad vs bench >0% genera valor    <-1%
    Beta        Sensibilidad al benchmark       <0.5 RF  ~1 RV      >1.5
    R²          Explicación del benchmark       >60% representativo <20%
    Omega       Relación ganancias/pérdidas     >1.5 bueno          <1.0
    Skewness    Asimetría (derecha=positivo)   >0 mejor            <-0.5
    Kurtosis    Colas extremas vs normal        ~0 mejor            >2
    Tail Ratio  P95/|P5| de retornos            >1.2 mejor          <0.8
    Años BW     % años con drawdown >50%        <20% mejor          >40%

  ── INTERPRETACIÓN DE MÉTRICAS AVANZADAS ─────────────────────────────────────

    SHARPE RATIO (Ratio de Sharpe):
      Fórmula: (Rentabilidad - Rf) / Volatilidad
      ├─ >1.0  ✓ Excelente rendimiento ajustado al riesgo
      ├─ 0.5-1.0  ○ Bueno, pero hay margen de mejora
      ├─ 0-0.5  △ Moderado, el riesgo no está justificado
      └─ <0  ✗ Rentabilidad menor que Rf (evitar)
      
      Uso: Comparar fondos del mismo tipo (RF con RF, RV con RV).
           No comparar RF vs RV directamente (la pendiente es diferente).

    SORTINO RATIO (Ratio de Sortino):
      Fórmula: (Rentabilidad - Rf) / Volatilidad_bajista
      └─ Ignora volatilidad positiva (subidas), solo penaliza bajadas.
      └─ >1.0  ✓ Muy recomendable, especialmente para fondos conservadores
      └─ Más relevante que Sharpe para RF y fondos defensivos
      └─ CVaR es la evolución moderna de este concepto

    OMEGA RATIO (Ratio Omega):
      Fórmula: Σ(ganancias - umbral) / Σ(umbral - pérdidas)
      └─ >1.5  ✓ Excelente (ganancias superan pérdidas en proporción 1.5:1)
      └─ 1.0-1.5  ○ Bueno
      └─ <1.0  ✗ Más pérdidas que ganancias (evitar)
      └─ Más intuitivo que Sharpe para inversores con aversión al riesgo

    SKEWNESS (Asimetría):
      └─ >0  ✓ Distribución sesgada a la derecha (colas de ganancias)
      └─ ~0  ○ Distribución simétrica (ideal en teoría)
      └─ <-0.5  ✗ Sesgada a la izquierda (colas de pérdidas, evitar)
      └─ En crisis: aumenta hacia -1 (distribución muy asimétrica hacia pérdidas)

    KURTOSIS (Curtosis):
      └─ ~0  ✓ Distribución similar a la normal (sin sorpresas extremas)
      └─ >2  ⚠ Colas gruesas (más sorpresas negativas de lo esperado)
      └─ <-1  ✓ Colas ligeras (menos sorpresas, pero también pocas ganancias)
      └─ En fondos monetarios: kurtosis alto indica cambios bruscos de calidad

    TAIL RATIO (Relación de colas):
      Fórmula: P95 (mejor mes) / |P5| (peor mes)
      └─ >1.2  ✓ Mejor, colas de ganancias más gruesas que pérdidas
      └─ ~1.0  ○ Simétrico
      └─ <0.8  ✗ Peor, colas de pérdidas más gruesas (asymmetric risk)
      └─ Durante crisis: converge hacia 0.5 en RV

    VaR (Value at Risk):
      └─ VaR 95% mensual = -3%  →  en 1 de cada 20 meses, pérdida ≥ 3%
      └─ Para RF: <-2% mensual es alto
      └─ Para RV: <-5% mensual es bajo, >-8% es alto
      └─ CVaR (Expected Shortfall): media de las pérdidas en el peor 5%
      └─ CVaR siempre > VaR en valor absoluto (más conservador)

    BETA (Sensibilidad al benchmark):
      └─ <0.5  ✓ Muy defensivo (mitiga bajadas del mercado)
      └─ 0.5-1.0  ○ Moderadamente defensivo
      └─ ~1.0  ○ Refleja el mercado
      └─ 1.0-1.5  △ Amplificador (sube/baja más que el mercado)
      └─ >1.5  ✗ Alto apalancamiento o estrategia especulativa

    ALPHA (Exceso de rentabilidad):
      └─ >0%  ✓ El gestor genera valor extra respecto al benchmark
      └─ ~0%  ○ Gestor sigue el benchmark (seguidor o muy eficiente)
      └─ <-0%  ✗ El gestor destruye valor (incluso con gestión pasiva hubiera ido mejor)
      └─ En fondos pasivos (ETFs): Alpha ~0 por diseño (replicar índice)

    R² (Bondad de ajuste):
      └─ >80%  ✓ Benchmark es muy representativo del fondo
      └─ 60-80%  ○ Benchmark es representativo
      └─ 40-60%  △ Benchmark parcialmente representativo
      └─ <40%  ✗ Benchmark muy diferente (considerar otro o sin benchmark)
      └─ En fondos con múltiples activos: R² bajo es esperado

  ── GRÁFICO SUBACUÁTICO (UNDERWATER CHART) ───────────────────────────────────

    Panel superior: NAV base 100 (azul) + drawdown continuo (rojo).
                    Zona roja = tiempo "bajo el agua" respecto al máximo histórico.
                    Cada vez que NAV cae bajo su máximo anterior, el área roja crece.

    Panel inferior izq.: Histograma retornos mensuales (verde/rojo) + curva normal
                        teórica (azul discontinuo).
                        Si las colas son más gruesas que la curva: más sorpresas
                        extremas de lo esperado.
                        P5 (naranja) = peor mes típico
                        P95 (azul) = mejor mes típico

    Panel inferior der.: % días bajo el agua por año.
                        Verde <20%   → excelente (renta fija/defensivos)
                        Naranja 20-50% → normal (renta variable moderada)
                        Rojo >50%   → preocupante (fondos muy volátiles)
                        
                        Un fondo conservador no debería superar 20% en ningún año.
                        En crisis 2008-2009: muchos fondos >80% bajo el agua.

    Interpretación combinada:
      Si Panel sup. tiene muchas "islas" roja → múltiples recuperaciones de máximos
      Si Panel inf.izq. tiene colas muy gruesas → distribución no es normal
      Si Panel inf.der. tiene barras rojas → periodos prolongados de pérdidas

  ── FRONTERA EFICIENTE (opción 8) ────────────────────────────────────────────

    Cada punto de la nube = una cartera simulada (8.000 simulaciones).
    Color del punto = Sharpe (verde=alto, rojo=bajo).
    
    ★ Cartera de Sharpe máximo  → mejor relación rentabilidad/riesgo
                                  (punto más al noroeste de la nube).
    ◆ Cartera de mínima varianza → menor riesgo posible con estos activos
                                  (punto más a la izquierda).
    ● Cartera equi-ponderada     → referencia igualitaria (1/N pesos).
    ◇ Cartera actual             → composición presente de tu cartera.

    Lectura:
    ├─ Si Cartera actual está lejos (SE) de ★ → hay margen de mejora
    ├─ Si Cartera actual está cerca de ◆ → muy conservadora
    ├─ Si Cartera actual está en la nube densa → buena diversificación
    └─ Si Cartera actual está aislada → concentrada en pocos activos

    Caveats:
    ├─ La frontera no considera fiscalidad ni restricciones de liquidez
    ├─ Supone que la correlación histórica se mantiene (no es cierto en crisis)
    ├─ Los retornos esperados futuros pueden diferir de los históricos
    └─ Considera cambios de pesos, no rebalanceos periódicos (costes)

  ── VaR Y CVaR (opción 8) ────────────────────────────────────────────────────

    Tres métodos calculados:

    1. VaR Histórico (95%):
       └─ Percentil 5 de los retornos históricos mensual.
       └─ Más robusto en distribuciones no-normales.
       └─ Para cartera RF de renta fija:
          ├─ VaR mensual < -1%  ✓ Muy bajo (monetarios)
          ├─ VaR mensual -1% a -3%  ○ Moderado (RF corta/media)
          ├─ VaR mensual -3% a -5%  △ Elevado (RF larga)
          └─ VaR mensual < -5%  ✗ Revisar composición

    2. VaR Paramétrico (normal):
       └─ Supone distribución normal: Retorno_medio - 1.645*Volatilidad (para 95%)
       └─ Subestima el riesgo si hay colas gruesas (kurtosis > 0)
       └─ Pero es rápido de calcular y actualizar

    3. CVaR / Expected Shortfall (media de pérdidas en peor 5%):
       └─ Siempre mayor en valor absoluto que VaR.
       └─ Más conservador y recomendado por reguladores (Basilea III)
       └─ Para tomar decisiones de riesgo extremo: usar CVaR, no VaR.

    Ejemplo de interpretación:
    ├─ VaR 95% mensual = -2.5%  →  en el peor mes de 20 (5%), pérdida ≥ 2.5%
    ├─ CVaR 95% mensual = -3.8%  →  media de pérdidas en el 5% peor de los meses
    ├─ Si cartera = 100.000 €:
    │  ├─ Riesgo normal (VaR): -2.500 €
    │  └─ Riesgo extremo (CVaR): -3.800 €
    └─ En crisis, diferencia crece (VaR subestima mucho)

  ── CORRELACIONES (opción 5) ─────────────────────────────────────────────────

    Correlación estática (período completo):
      >0.80  ✗ Alta correlación, diversificación limitada
      0.40–0.80  ○ Correlación moderada, diversificación parcial
      0.00–0.40  ✓ Baja correlación, buena diversificación real
      <0.00  ✓✓ Correlación negativa (ideal, se compensan las bajadas)

    Correlación rodante (ventana 60 días):
      └─ Muestra cómo evolucionan las correlaciones en el tiempo
      └─ Si sube >0.30 en pocas semanas → posible co-movimiento o evento de mercado
      └─ En crisis las correlaciones convergen hacia +1 → diversificación "falla"
      └─ La banda punteada marca desviación estándar histórica
      └─ Si sale de la banda → comportamiento anómalo

    Matriz de calor (heatmap):
      └─ Células verdes (correlación negativa) → diversificación excelente
      └─ Células amarillas (correlación ~0.5) → diversificación buena
      └─ Células rojas (correlación >0.8) → diversificación pobre

    Detección de crisis por correlación:
      └─ Si MUCHAS correlaciones pares suben simultáneamente
      └─ Y convergen hacia +1 en pocas semanas
      └─ → Probable event de estrés sistémico (crisis de mercado)
      └─ Acciones a tomar: revisar VaR, aumentar RF, vender volátiles

  ── ANÁLISIS TÉCNICO (opción 7) ──────────────────────────────────────────────

    Indicador   Fórmula/Descripción             Señal +             Señal -
    ─────────────────────────────────────────────────────────────────────────
    SMA         Promedio móvil simple           Cruce dorado        Cruce muerte
                (20, 50, 200 días)              (50>200)            (50<200)
    
    Bollinger   Media ± 2*Desv.Est. (20d)       NAV en banda inf.   NAV en banda sup.
    Squeeze     Bandwidth <P10 histórico        Calma antes tormenta —
    
    MACD        (EMA12 - EMA26) vs EMA9 señal   Cruce alcista       Cruce bajista
                Histograma = diferencia        (L>S, histogram>0)  (L<S, histogram<0)
    
    ATR         Rango verdadero promedio        Bajo (volatilidad ↓) Alto (volatilidad ↑)
                (14 días)                      = oportunidad       = riesgo
    
    RSI         Fuerza relativa (14 días)       Divergencia alcista Divergencia bajista
                (0-100 escala)                 (RSI bajo, NAV ↑)   (RSI alto, NAV ↓)
                                               O RSI 30–70 neutral >80 o <20 extremo

    Aplicabilidad según tipo de fondo:
    ├─ RF ultra corto / Monetario: ESCASA UTILIDAD (volumen muy bajo)
    ├─ RF corto-medio plazo: BAJA-MEDIA (menos ruido que acciones)
    ├─ RF largo plazo: MEDIA (máxima sensibilidad a tasas de interés)
    ├─ Renta variable: ALTA (ruido técnico, reversiones, momentum)
    └─ Acciones individuales: ALTA (máxima especulación)

    Estrategia de trading técnico:
    └─ Usar MÚLTIPLES indicadores (no solo uno)
    └─ Esperar CONFIRMACIÓN entre indicadores
    └─ Ejemplo: Cruce alcista MACD + RSI <70 + SMA bullish
    └─ Usar ATR para ajustar stop-loss (p.ej. -2*ATR desde entrada)
    └─ Nunca ignorar el fundamento (técnico es corto plazo)

    Limitaciones:
    ├─ Profecía autocumplida: todos usan los mismos indicadores
    ├─ Genera falsos positivos en consolidaciones laterales
    ├─ No funciona en mercados sin tendencia (range-bound)
    ├─ Máxima efectividad en mercados trending y volátiles
    └─ Stop-loss técnico puede ser ejecutado por ruido intradiario

  ── HEATMAP MENSUAL (CALENDAR HEATMAP) ───────────────────────────────────────

    Estructura: Filas = Años, Columnas = Meses (Ene-Dic)
    Color de cada celda = Retorno mensual (verde=positivo, rojo=negativo)
    Intensidad del color = Magnitud del retorno

    Lecturas útiles:
    ├─ Buscar patrones estacionales (p.ej. enero fuerte, verano débil)
    ├─ Identificar años problemáticos (fila muy roja)
    ├─ Meses recurrentemente flojos (columna con más rojo)
    ├─ Evolución temporal (¿empeora en años recientes?)
    └─ Simetría de retornos (¿más positivos que negativos?)

    Para gestores:
    ├─ Gestor A: Heatmap verde mayormente → valor añadido consistente
    ├─ Gestor B: Alternancia verde/rojo → rentabilidad errática
    ├─ Si gestor A bate benchmark > 60% de meses → Alpha real


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
4. DEPENDENCIAS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ── INSTALACIÓN ──────────────────────────────────────────────────────────────

    pip install yfinance openpyxl pandas matplotlib scipy

    Paquete         Versión mín.  Para qué se usa
    ──────────────────────────────────────────────────────────────────────────
    pandas          2.0           DataFrames, resampleo, retornos, time series
    yfinance        0.2           Descarga de cotizaciones de Yahoo Finance
    openpyxl        3.1           Lectura/escritura de ficheros .xlsx con estilos
    matplotlib      3.7           Gráficas (base 100, técnico, subacuático, heatmap)
    scipy           1.11          Optimización Markowitz, VaR paramétrico,
                                  distribución normal, ajuste de curvas
    numpy           (con pandas)  Cálculos vectoriales de métricas
    reportlab       (opcional)    Generación de PDF avanzada (futuro)

  ── INSTALACIÓN EN DIFERENTES ENTORNOS ───────────────────────────────────────

    En Linux/Mac terminal:
      $ pip install --upgrade pip
      $ pip install yfinance openpyxl pandas matplotlib scipy

    En Anaconda Prompt (Windows):
      > conda install -c conda-forge yfinance openpyxl pandas matplotlib scipy

    En Google Colab (ejecutar en celda):
      !pip install yfinance openpyxl --upgrade

    En Jupyter sin conda:
      import sys
      !{sys.executable} -m pip install yfinance openpyxl scipy --upgrade

  ── ENTORNOS SOPORTADOS ──────────────────────────────────────────────────────

    · Python 3.10, 3.11, 3.12 (3.13 en desarrollo)
    · Jupyter Notebook / JupyterLab (recomendado para uso interactivo)
    · Google Colab (monta Google Drive automáticamente)
    · Terminal / línea de comandos (modo texto sin gráficas interactivas)
    · Anaconda / conda — instalación recomendada para usuarios no técnicos
    · VSCode con extensión Jupyter

    Instalación recomendada para usuarios sin experiencia:
      1. Descargar Anaconda desde https://www.anaconda.com
      2. Abrir Anaconda Navigator → crear entorno Python 3.11
      3. Lanzar Jupyter Notebook desde Navigator
      4. En terminal de Jupyter: pip install yfinance openpyxl scipy

    Para compatibilidad máxima:
      pip install pandas>=2.0 yfinance>=0.2 openpyxl>=3.1 scipy>=1.11 matplotlib>=3.7

  ── DATOS EXTERNOS ───────────────────────────────────────────────────────────

    Yahoo Finance (gratuito, sin clave):
      └─ Cotizaciones históricas de fondos, ETFs, acciones e índices
      └─ Limitación: fondos monetarios publicados sin cupones (NAV plano)
      └─ El programa detecta y avisa este caso automáticamente
      └─ Latencia ~15-20 min para datos intradiarios
      └─ Datos EOD (fin de día) disponibles al día siguiente

    Datos macroeconómicos (futuro):
      └─ FRED (Federal Reserve Economic Data) para tasas, inflación, paro
      └─ Se obtendrá en: https://fred.stlouisfed.org/docs/api/api_key.html
      └─ Clave de API gratuita para usuarios registrados
      └─ Versión futura integrará automáticamente

    No se requiere clave de API para las funcionalidades actuales.

  ── FICHEROS DE DATOS ────────────────────────────────────────────────────────

    Formato CSV esperado en Datos_CSV/:
      ├─ Columna obligatoria : Date  (formato YYYY-MM-DD o similar)
      ├─ Columna de precio   : Adj Close  (preferida) o  Close
      ├─ Generado por el programa automáticamente al descargar de Yahoo Finance
      └─ Estructura esperada:
            Date,       Close,    Adj Close,    Volume
            2024-01-02, 123.45,   123.45,       1000000
            2024-01-03, 124.10,   124.10,       900000
            ...

    Para planes de pensiones o fondos sin ticker Yahoo:
      ├─ Crear manualmente un CSV con las columnas Date y Close
      ├─ NAV unitario diario (NO el valor total de la posición)
      ├─ Formato fecha: YYYY-MM-DD
      └─ Ejemplo:
            Date,       Close
            2024-01-02, 50.123
            2024-01-03, 50.456
            ...

    Validación automática del programa:
      ├─ Detecta columnas faltantes
      ├─ Convierte fechas a formato estándar
      ├─ Rellena huecos (forward fill si <5%, error si >5%)
      └─ Avisa si hay datos duplicados (por fecha)

  ── REQUISITOS DE SISTEMA ────────────────────────────────────────────────────

    Espacio en disco:
      ├─ Programa principal: 500 KB
      ├─ Datos para 50 fondos (5 años): ~100 MB
      ├─ Excels de carteras: <1 MB cada uno
      ├─ PDFs exportados: 2-5 MB cada uno
      └─ Total recomendado: 500 MB

    Memoria RAM:
      ├─ Mínimo: 2 GB (análisis de carteras pequeñas <100 fondos)
      ├─ Recomendado: 4-8 GB (para 100-500 fondos, cálculo Markowitz)
      ├─ Simulación Markowitz (8.000 carteras): ~200 MB temporal
      └─ No es paralizable (uso single-thread)

    Conexión a internet:
      ├─ Primera descarga de todos los fondos: 30-60 min dependiendo de volumen
      ├─ Actualización semanal de 20 fondos: 5-10 min
      ├─ Ancho de banda: ~50 MB por 100 fondos/5 años
      └─ Se puede ejecutar offline si los CSVs ya están descargados


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
5. METODOLOGÍA Y REFERENCIAS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ── FUENTES TEÓRICAS ─────────────────────────────────────────────────────────

    Teoría Moderna de Carteras (MPT — Markowitz 1952):
      └─ Referencias en código: frontera eficiente, pesos óptimos
      └─ Libro: "Portfolio Selection" — Harry Markowitz
      └─ Limitaciones: asume normalidad de retornos (en crisis no se cumple)

    Capital Asset Pricing Model (CAPM):
      └─ Fórmula: E(Rx) = Rf + β(E(Rm) - Rf)
      └─ Usado para calcular Alpha: Rx - [Rf + β(Benchmark - Rf)]
      └─ Referencias: William Sharpe (1964)
      └─ Limitaciones: relación lineal beta-retorno (simplista)

    Análisis de Riesgo de Colas (Tail Risk):
      └─ CVaR / Expected Shortfall como mejora sobre VaR
      └─ Método de valor en riesgo condicional
      └─ Regulación: Basilea III (instituciones financieras)
      └─ Acevedo et al. "Conditional Value-at-Risk" (2002)

    Análisis Técnico:
      └─ Elliott Wave, Dow Theory, patrones de velas
      └─ MACD: Gerald Appel (1979)
      └─ Bandas de Bollinger: John Bollinger (1983)
      └─ RSI: J. Welles Wilder Jr. (1978)
      └─ Crítica académica: "A Random Walk Down Wall Street" (Burton Malkiel)

    Metodología Morningstar:
      └─ Sharpe y Sortino calculados con retornos MENSUALES
      └─ Tasa libre de riesgo = Euribor 3M del período
      └─ Comparación con fondos peer de la misma categoría
      └─ Rating de estrellas basado en Morningstar Risk-Adjusted Return

  ── DECISIONES DE DISEÑO EN EL CÓDIGO ────────────────────────────────────────

    1. Retornos logarítmicos vs simples:
       └─ Se usa LOG porque: mejor comportamiento en compounding,
          propiedades estadísticas más limpias, menos sesgo en distribución.
       └─ Fórmula: r = ln(P_t / P_{t-1})

    2. Volatilidad anualizada:
       └─ Fórmula: σ_anual = σ_diaria × √252 (días de trading/año)
       └─ No es: σ_diaria × 252 (eso sería error común)

    3. Sharpe Ratio:
       └─ Usa Euribor 3M trimestral como Rf
       └─ Se calcula con RETORNOS MENSUALES, no diarios
       └─ Motivo: Morningstar standard, menos ruido en datos mensuales

    4. Beta:
       └─ Regresión lineal: Fondo vs Benchmark (mínimos cuadrados)
       └─ Ventana: últimos 3 años si disponible, sino máximo histórico
       └─ Interpretación: β=1.2 significa sube 12% si benchmark sube 10%

    5. Alpha:
       └─ Fórmula: α = (Retorno_fondo) - [Rf + β(Retorno_benchmark - Rf)]
       └─ Anualizado: α_anual = α_mensual × 12
       └─ Positivo = gestor agrega valor, Negativo = destruye valor

    6. Max Drawdown:
       └─ Máxima caída DESDE UN MÁXIMO hasta cualquier mínimo posterior
       └─ NO es la diferencia entre máximo y mínimo del período
       └─ Fórmula: DD = (Precio_min - Precio_max) / Precio_max

    7. VaR Histórico:
       └─ Percentil 5 de retornos mensuales históricos
       └─ Se ordena: -sorteo retornos -coge el percentil P5
       └─ Ventaja: no asume distribución normal
       └─ Desventaja: necesita muchos datos (~5 años mínimo)

    8. CVaR / Expected Shortfall:
       └─ Media de retornos PEORES que el VaR
       └─ Siempre > VaR en valor absoluto (más conservador)
       └─ No asume normalidad
       └─ Recomendado por reguladores modernos

    9. Frontera Eficiente:
       └─ Monte Carlo: 8.000 carteras aleatorias
       └─ Para cada una: calcula Sharpe y almacena
       └─ Luego: optimización con scipy.optimize.minimize para ★ y ◆
       └─ Razón: Monte Carlo más robusto en n activos > 5

    10. Análisis Técnico:
        └─ SMA con pandas.rolling().mean()
        └─ MACD: EMA12 - EMA26, señal = EMA9
        └─ ATR: Range true = max(H-L, |H-Close_prev|, |L-Close_prev|)
        └─ RSI: RS = (Gains_avg / Loss_avg), RSI = 100 - (100/(1+RS))

  ── LIMITACIONES Y CAVEATS ───────────────────────────────────────────────────

    Limitaciones de datos:
    ├─ Fondos monetarios: NAV no incluye cupones acumulados → rentabilidad incorrecta
    ├─ Yahoo Finance: latencia 15-20 min en datos, puede haber gaps
    ├─ Fondos jóvenes (<1 año): insuficientes datos para métricas robustas
    ├─ ETFs sintéticos (swap): tracking error puede ser alto, no captado aquí
    └─ Fondos cerrados: pueden tener spreads bid-ask superiores a lo normal

    Limitaciones teóricas:
    ├─ Markowitz: asume normalidad (en crisis hay colas gruesas) → VaR subestimado
    ├─ CAPM: relación lineal β-retorno (en realidad hay estructuras más complejas)
    ├─ Correlaciones: supone estables (en crisis convergen todas hacia +1)
    ├─ Bench mark: único vs múltiples factores (Fama-French sería mejor)
    └─ Retornos pasados: no garantizan futuros ("past performance is not indicative...")

    Limitaciones prácticas:
    ├─ Costes de transacción: frontera eficiente no los considera
    ├─ Fiscalidad: no calcula IRPF/impuestos sobre ganancias
    ├─ Liquidez: supone venta inmediata al precio actual
    ├─ Restricciones: no permite posiciones cortas (short selling)
    ├─ Rebalanceo: frontera es estática, no dinámicas
    └─ Cambio de divisa: todos los CSVs en misma moneda (ojo con internacionales)

  ── CÓMO COMBATIR ESTOS PROBLEMAS ────────────────────────────────────────────

    1. Datos de fondos monetarios:
       └─ Solución: obtener NAV+cupones acumulados del folleto del fondo
       └─ O: usar iShares Money Market ETF (VMVX) como proxy

    2. Insuficientes datos:
       └─ Solución: para fondos <1 año usar Benchmark proxy (peer group)
       └─ O: esperar a 1 año de datos para confianza estadística

    3. Markowitz subestima riesgo en crisis:
       └─ Solución: aumentar VaR teórico 1.5x-2x para stress testing
       └─ O: usar modelo de colas pesadas (Student-t distribution)

    4. Correlaciones cambian en crisis:
       └─ Solución: usar correlación rodante (ventana 60d)
       └─ O: tener cartera con activos con correlación negativa acreditada (oro, bonos)

    5. No considerar costes:
       └─ Solución: restar comisiones a retornos calculados (α_ajustado)
       └─ Fórmula: α_real = α_calculado - comisión_anual(%)

    6. Fiscalidad:
       └─ Solución: calcular IRPF = ganancias * 19% (España)
       └─ Rentabilidad neta = rentabilidad_bruta - IRPF
       └─ (Pero esto requiere interacción con sistema tributario)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
6. EJEMPLOS DE USO AVANZADO
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ── SCENARIO 1: INVERSOR CONSERVADOR CON FONDO MONETARIO ──────────────────────

    Objetivo: Verificar que la cartera genera caja sin sorpresas.
    
    Pasos:
    1. Descargar cotizaciones (opción 1)
    2. Actualizar cartera (opción 2)
    3. Revisar métricas (opción 4):
       └─ Buscar Volatilidad < 0.5% (normal)
       └─ Sharpe > 0.5 (mínimo esperado)
       └─ Si vol > 1% o Sharpe < 0 → problemas de calidad crediticia
    4. Revisar gráfico subacuático (opción 7):
       └─ % días bajo el agua debería ser <5% (fondos monetarios)
       └─ Si aparecen islas de rojo extensas → cambios de calidad

    Acción recomendada:
    └─ Si métricas normales: hodl (mantener posición sin cambios)
    └─ Si volatilidad anormal: revisar composición (hay de crédito)
    └─ Si downgrade de rating: considerar salida
    └─ Rebalanceo: no necesario (posición defensiva estática)

  ── SCENARIO 2: INVERSOR MODERADO CON CARTERA BALANCEADA 50/50 ────────────────

    Objetivo: Verificar que la cartera mantiene equilibrio riesgo/retorno.
    
    Pasos:
    1. Actualizar cartera (opción 2)
    2. Ver correlaciones (opción 5):
       └─ RF vs RV correlación debería ser 0.30-0.50 (buena diversificación)
       └─ Si >0.70 → mercado acoplado, comprar activos descorrelacionados
       └─ Si <0.20 → poco movimiento de RV (mercado plano o deflacionista)
    3. Análisis agregado (opción 8):
       └─ Frontera eficiente: ¿está la cartera cerca de ★?
       └─ VaR mensual: ¿está dentro de tolerancia (p.ej. -3%)?
       └─ Contribución riesgo: ¿RF <30%, RV >70% del riesgo?
    4. Exportar PDF (opción 6): informe para asesor

    Acción recomendada:
    └─ Si Sharpe <0.5 para cartera balanceada → ineficiente, rebalancear
    └─ Si correlación RF/RV sube >0.70 → evento sistémico, aumentar RF
    └─ Si VaR sale de límite → reducir RV
    └─ Rebalanceo anual (trimestral en volatilidad alta)

  ── SCENARIO 3: GESTOR DE CARTERAS MULTICLIENTE ──────────────────────────────

    Objetivo: Comparar desempeño de múltiples carteras vs benchmark.
    
    Pasos:
    1. Cargar cartera cliente A (opción B)
    2. Ver métricas (opción 4) y Alpha vs benchmark:
       └─ Repetir para clientes B, C, D...
    3. Consolidar resultados en Excel exportado (opción 6)
    4. Identificar fondos con Alpha positivo consistente (>0.5% anual)
    5. Identificar fondos con correlación alta con benchmark (R² > 90%):
       └─ Candidatos para cambio a fondos indexados (menor comisión)

    KPIs monitoreados:
    ├─ Alpha medio por cartera: media ponderada de αs de fondos
    ├─ % fondos batiendo benchmark: >50% es signo de valor
    ├─ Volatilidad vs benchmark: si cartera es más volátil con mismo α → ineficiente
    ├─ Ratio Sharpe cartera: >0.7 es objetivo mínimo
    └─ Máximo drawdown vs benchmark: cartera no debe ser peor

    Reportes:
    └─ Mensual: Alpha, Sharpe, Drawdown por cartera
    └─ Trimestral: análisis técnico, heatmaps estacionales
    └─ Anual: revisión de benchmarks, rotación de fondos

  ── SCENARIO 4: TRADER TÉCNICO EN ACCIONES ───────────────────────────────────

    Objetivo: Identificar señales técnicas para entrada/salida corto plazo.
    
    Pasos:
    1. Cargar acción individual (p.ej. AAPL) (opción 1)
    2. Análisis técnico avanzado (opción 7):
       └─ SMA: ¿está en cruce dorado (50>200)?
       └─ MACD: ¿acaba de girar positivo?
       └─ Bollinger: ¿está en banda inferior (rebote)?
       └─ RSI: ¿está <30 (sobreventa)?
    3. Si MÚLTIPLES indicadores confirman → entrada larga
    4. Poner stop-loss 2*ATR bajo entrada
    5. Objetivo: 1.5-2x del ATR de ganancia

    Trade setup ejemplo:
    ├─ Precio NAV 150 €
    ├─ ATR(14) = 3 €
    ├─ Entrada larga: cuando Cruce dorado SMA (50>200) + RSI gira >30
    ├─ Stop loss: 150 - 2*3 = 144 € (riesgo 6 €)
    ├─ Target: 150 + 1.5*3 = 154.5 € (ganancia 4.5 €)
    ├─ Risk/Reward: 6/4.5 = 1.33x (aceptable)
    └─ Esperar confirmación de volumen

    Limitación:
    └─ Técnico sin fundamental = casino (máximo uso 10-20% cartera)

  ── SCENARIO 5: ANÁLISIS POST-CRISIS (ESTRÉS SISTÉMICO) ──────────────────────

    Objetivo: Cuantificar impacto de shock de mercado.
    
    Pasos:
    1. Descarga de datos: activar rango largo (5-10 años)
    2. Correlación rodante (opción 5):
       └─ Buscar período de crisis (2008, 2011, 2020, 2022)
       └─ Observar convergencia todas correlaciones → +0.9-1.0
       └─ Duración del pico (semanas, meses?)
    3. Gráfico subacuático (opción 7):
       └─ % días bajo el agua en 2008 → probablemente >60-80%
       └─ Drawdown máximo: típicamente -40% a -60% en acciones
    4. VaR histórico (opción 8):
       └─ Construir escenarios:
       └─ "¿Si repite 2008, cuál es máxima pérdida?"
       └─ Respuesta: máximo drawdown histórico
    5. Stress test:
       └─ "¿Si cae 20% más de lo peor visto?" → aplicar factor 1.2x al drawdown

    Acciones recomendadas:
    ├─ Si cartera tiene correlación RF/RV < 0.5: buena diversificación
    ├─ Si tiene > 0.7: cambiar a fondos decorrelacionados (oro, bonos LT)
    ├─ Aumentar % RF: cada 1% más de RF reduce drawdown ~0.5-1%
    ├─ Considerar futuros o opciones de protección (VIX)
    └─ Rebalanceo: contraciclíco (vender ganadores, comprar perdedores)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
7. AVISO LEGAL
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Este programa es una herramienta de análisis personal y seguimiento de
  carteras de inversión. No constituye asesoramiento financiero, fiscal ni
  legal de ningún tipo.

  Los resultados que genera —métricas, gráficas, diagnósticos, sugerencias
  de pesos, VaR, frontera eficiente— son puramente informativos y no deben
  interpretarse como recomendaciones de compra, venta o mantenimiento de
  ningún instrumento financiero.

  El autor no se responsabiliza de las decisiones de inversión que el usuario
  pueda tomar basándose en los resultados del programa, ni de las pérdidas
  patrimoniales que pudieran derivarse de dichas decisiones.

  Los datos de cotizaciones se obtienen de Yahoo Finance y pueden diferir
  de los datos oficiales publicados por las gestoras, depositarios o
  mercados regulados. Yahoo Finance no garantiza la exactitud, integridad
  ni actualidad de sus datos.

  Para fondos monetarios, Yahoo Finance no incluye los cupones en el NAV
  publicado, lo que produce métricas de rendimiento y riesgo incorrectas.
  El programa detecta esta situación y emite un aviso, pero no corrige
  los datos automáticamente.

  En España: Este programa no está regulado por la CNMV. Su uso no sustituye
  la asesoramiento de un profesional financiero certificado (SNMV/CNMV).

  Consulte siempre a un asesor financiero profesional regulado (CNMV en
  España) antes de tomar cualquier decisión de inversión relevante.

  USO BAJO RESPONSABILIDAD PROPIA: El usuario acepta que usa este programa
  enteramente bajo su responsabilidad. El autor rechaza toda responsabilidad
  civil, penal o administrativa derivada del uso de este programa.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
8. LICENCIA DE USO
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Creative Commons BY-NC 4.0 — Atribución – No Comercial

  Este proyecto y su documentación se distribuyen bajo licencia
  Creative Commons BY-NC 4.0 (Atribución – No Comercial).

  SE PERMITE:
    · Usar, copiar y adaptar el código con fines educativos o personales.
    · Compartir versiones modificadas citando la fuente original.
    · Integrar en proyectos personales o académicos sin ánimo de lucro.
    · Enseñanza y divulgación (blogs, tutoriales, universidades).

  NO ESTÁ PERMITIDO:
    · Uso comercial del código o derivados del mismo.
    · Redistribución con fines lucrativos o en productos de pago.
    · Integración en plataformas de asesoramiento automatizado (robo-advisors).
    · Uso por fondos, gestoras u otros intermediarios financieros.
    · Venta del código o documentación asociada.
    · Eliminar o modificar los avisos de autoría y licencia.

  Texto completo de la licencia:
    https://creativecommons.org/licenses/by-nc/4.0/deed.es

  Cita sugerida:
    "Gestor de Cartera de Valores en Python — actualizar_cartera.py
     Versión 2.0 · Publicado en [plataforma] bajo licencia CC BY-NC 4.0"

  Preguntas frecuentes sobre la licencia:
    P: ¿Puedo usar esto en mi startup financiera?
    R: No, es NO-COMERCIAL. Si esperas generar ingresos, solicita licencia
       por separado.

    P: ¿Puedo enseñarlo en mi universidad?
    R: Sí, educación es permitido. Cita la fuente.

    P: ¿Puedo hacer una API REST con este código?
    R: No, eso sería comercial. Pregunta al autor.

    P: ¿Puedo traducir la documentación a otro idioma?
    R: Sí, siempre que cites y mantengas la licencia CC BY-NC.

    P: ¿Puedo usarlo si los datos son de acceso libre?
    R: El dato acceso libre no revoca la licencia. La restricción es sobre
       el código/programa, no sobre los datos.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
9. ROADMAP Y MEJORAS FUTURAS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Versión 2.1 (Próximo mes):
  ├─ Soporte para fondos de fondos (FoF)
  ├─ Cálculo de beta rolling (ventana móvil 60d)
  ├─ Gráfico de contribución de retorno por activo
  └─ Exportación a CSV con todas las métricas

  Versión 2.2 (3 meses):
  ├─ Integración con FRED (datos macroeconómicos)
  ├─ Correlación con tasa de cambio EUR/USD
  ├─ Factor models (Fama-French 5 factores)
  ├─ Análisis de performance attribution (Brinson-Fachler)
  └─ Alertas automáticas (email/Telegram)

  Versión 3.0 (6 meses):
  ├─ Machine learning: predicción de correlaciones (LSTM)
  ├─ Optimización con restricciones (mínimo/máximo por activo)
  ├─ Backtesting de estrategias técnicas
  ├─ Interfaz gráfica (PyQt o Streamlit)
  ├─ Base de datos (SQLite o PostgreSQL) en lugar de CSVs
  └─ API REST para integración con brokers (Saxo, Degiro)

  Versión 3.1 (9 meses):
  ├─ Simulación de Montecarlo de trayectorias futuras
  ├─ Cálculo de capital necesario (subnormación)
  ├─ Integración fiscal (IRPF automático por jurisdicción)
  ├─ Análisis de cambio de divisa (FX hedging)
  └─ Comparación con robo-advisors del mercado

  Funcionalidades lejanas (futuro):
  ├─ Órdenes automáticas directas al broker
  ├─ Ejecutar trades basado en señales técnicas
  ├─ Rebalanceo automático
  ├─ Sincronización multi-dispositivo (cloud)
  └─ App móvil (iOS/Android)

  Contributing:
  └─ Para reportar bugs: github.com/[proyecto]/issues
  └─ Para sugerencias: github.com/[proyecto]/discussions
  └─ Para pull requests: ver CONTRIBUTING.md


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
10. SOPORTE Y CONTACTO
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Reportar bugs:
  └─ GitHub Issues: [enlace del proyecto]
  └─ Email: [email del autor]
  └─ Título: "[BUG] Descripción breve"

  Solicitar funcionalidad:
  └─ GitHub Discussions: [enlace del proyecto]
  └─ Incluir caso de uso y justificación

  Documentación online:
  └─ Wiki: [enlace]
  └─ Blog: [enlace]
  └─ Video tutoriales: [enlace YouTube]

  Comunidad:
  └─ Discord: [enlace]
  └─ Telegram: [enlace]
  └─ Reddit: r/[comunidad]

  Financiar el proyecto:
  └─ GitHub Sponsors: [enlace]
  └─ Patreon: [enlace]
  └─ Compra de café: [buymeacoffee]

  Preguntas frecuentes: ver FAQ.md en el repositorio


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
11. CHANGELOG
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  v2.0 (2026-05-02):
  ├─ Release inicial de versión 2.0
  ├─ Soporte para 7 clases modularizadas
  ├─ Análisis técnico avanzado (SMA, MACD, Bollinger, ATR, RSI)
  ├─ Frontera eficiente Markowitz (Monte Carlo 8.000 carteras)
  ├─ VaR histórico, paramétrico y CVaR
  ├─ Correlación estática y rodante
  ├─ Exportación PDF y Excel
  ├─ Gráfico subacuático con histograma de retornos
  ├─ Heatmap mensual y anual
  ├─ Soporte multi-entorno (Jupyter, Colab, CLI, Anaconda)
  ├─ Documentación completa y ejemplos
  └─ Licencia CC BY-NC 4.0

  v1.9 (2026-04-15):
  ├─ Corrección de bugs en descarga de Yahoo Finance
  ├─ Optimización de rendimiento en Markowitz (paralización)
  ├─ Mejora de manejo de excepciones
  └─ Actualización de dependencias

  v1.0 (2025-01-01):
  ├─ Versión inicial beta
  ├─ Gestión de carteras básica
  ├─ Métricas Sharpe, Sortino, máx drawdown
  └─ Exportación a Excel


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

                              FIN DE DOCUMENTACIÓN

  Última actualización: 2026-05-02
  Versión: 2.0
  Mantenidor: [autor]
  Licencia: CC BY-NC 4.0

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

## Clase 1 — GestorRutas
Detección de entorno, configuración de paths, Colab vs local. Unas 80 líneas.

In [2]:
import os
from pathlib import Path
from typing import Dict, List, Literal
import platform

class GestorRutas:
    """
    Gestiona rutas y detección de entorno (Colab vs Local).
    Resuelve automáticamente problemas de rutas y crea directorios necesarios.
    """
    
    def __init__(self, ruta_base: str = None):
        """
        Inicializa el gestor de rutas.
        
        Args:
            ruta_base: Ruta base del proyecto. Si es None, usa la actual.
        """
        self.entorno = self._detectar_entorno()
        self.rutas = self._configurar_rutas(ruta_base)
        self._crear_directorios()
    
    def _detectar_entorno(self) -> Literal["colab", "local"]:
        """Detecta si estamos en Google Colab o en local."""
        try:
            import google.colab
            return "colab"
        except ImportError:
            return "local"
    
    def _configurar_rutas(self, ruta_base: str = None) -> Dict[str, Path]:
        """Configura rutas según el entorno detectado."""
        if self.entorno == "colab":
            base_dir = Path("/content/drive/MyDrive/Proyecto_Cartera")
        else:
            # En local, usa la ruta proporcionada o la actual
            if ruta_base:
                base_dir = Path(ruta_base)
            else:
                base_dir = Path.cwd()
        
        rutas = {
            "base": base_dir,
            "datos_csv": Path("/home/enri/Py_Renta_4_2026/Datos_csv"),
            "exportaciones": base_dir / "Exportaciones",
            "cartera": base_dir / "Carteras",
            "logs": base_dir / "Logs",
            "notebook_actual": Path.cwd(),
        }
        
        return rutas
    
    def _crear_directorios(self) -> None:
        """Crea los directorios necesarios si no existen."""
        for nombre, ruta in self.rutas.items():
            if nombre != "datos_csv":  # No crear Datos_csv, ya existe
                try:
                    ruta.mkdir(parents=True, exist_ok=True)
                    print(f"✓ Directorio: {ruta}")
                except Exception as e:
                    print(f"⚠️  Error al crear {nombre}: {e}")
    
    def _verificar_ruta_datos_csv(self) -> bool:
        """Verifica que la carpeta Datos_csv exista y sea accesible."""
        ruta = self.rutas["datos_csv"]
        if ruta.exists() and ruta.is_dir():
            print(f"✓ Carpeta Datos_csv accesible: {ruta}")
            return True
        else:
            print(f"❌ ADVERTENCIA: Datos_csv no encontrada en {ruta}")
            return False
    
    def obtener_ruta(self, tipo: str) -> Path:
        """
        Obtiene una ruta específica.
        
        Args:
            tipo: 'base', 'datos_csv', 'exportaciones', 'cartera', 'logs', 'notebook_actual'
        
        Returns:
            Path: Ruta solicitada
        
        Raises:
            ValueError: Si el tipo de ruta es desconocido
        """
        if tipo not in self.rutas:
            raise ValueError(f"Tipo de ruta desconocido: {tipo}. Disponibles: {list(self.rutas.keys())}")
        return self.rutas[tipo]
    
    def listar_csvs(self, extensión: str = "*.csv") -> List[Path]:
        """
        Lista todos los archivos en la carpeta Datos_csv.
        
        Args:
            extensión: Patrón de búsqueda (default: *.csv)
        
        Returns:
            Lista de rutas a archivos encontrados
        """
        ruta_csv = self.rutas["datos_csv"]
        if not ruta_csv.exists():
            print(f"❌ Carpeta no existe: {ruta_csv}")
            return []
        
        archivos = sorted(list(ruta_csv.glob(extensión)))
        print(f"📊 Encontrados {len(archivos)} archivos con patrón '{extensión}'")
        for archivo in archivos:
            print(f"   • {archivo.name}")
        return archivos
    
    def obtener_info_entorno(self) -> Dict:
        """Retorna información del entorno y rutas."""
        return {
            "entorno": self.entorno,
            "sistema_operativo": platform.system(),
            "usuario": os.getenv("USER", "desconocido"),
            "directorio_actual": str(Path.cwd()),
            "rutas": {k: str(v) for k, v in self.rutas.items()},
            "datos_csv_accesible": self._verificar_ruta_datos_csv()
        }
    
    def __repr__(self) -> str:
        return f"GestorRutas(entorno='{self.entorno}', datos_csv='{self.rutas['datos_csv']}')"

     

## Clase 2 — GestorCartera
Crear Excel, seleccionar cartera existente, leer/escribir datos, actualizar valores.   Unas 200 líneas.


In [3]:
import pandas as pd
from pathlib import Path
from typing import Optional, Dict, List
from datetime import datetime
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment

class GestorCartera:
    """
    Gestiona la creación, lectura y actualización de carteras en Excel.
    Maneja selección de carteras existentes y escritura de datos.
    """
    
    def __init__(self, gestor_rutas):
        """
        Inicializa el gestor de cartera.
        
        Args:
            gestor_rutas: Instancia de GestorRutas
        """
        self.gestor_rutas = gestor_rutas
        self.ruta_cartera = gestor_rutas.obtener_ruta("cartera")
        self.cartera_actual = None
        self.datos_cartera = None
    
    def listar_carteras_existentes(self) -> List[str]:
        """
        Lista todas las carteras Excel existentes.
        
        Returns:
            Lista con nombres de archivos .xlsx
        """
        if not self.ruta_cartera.exists():
            return []
        
        carteras = sorted([f.name for f in self.ruta_cartera.glob("*.xlsx")])
        print(f"📁 Carteras encontradas: {len(carteras)}")
        for i, cartera in enumerate(carteras, 1):
            print(f"   {i}. {cartera}")
        return carteras
    
    def crear_cartera_nueva(self, nombre_cartera: str, tickers: List[str], 
                           pesos: List[float] = None) -> Path:
        """
        Crea una nueva cartera en Excel.
        
        Args:
            nombre_cartera: Nombre del archivo (sin .xlsx)
            tickers: Lista de tickers
            pesos: Lista de pesos (si None, se distribuyen equitativamente)
        
        Returns:
            Path: Ruta del archivo creado
        """
        if pesos is None:
            pesos = [1/len(tickers) for _ in tickers]
        
        if not sum(pesos) > 0.99:  # Tolerancia para redondeos
            raise ValueError("Los pesos no suman aproximadamente 1.0")
        
        # Crear DataFrame
        df = pd.DataFrame({
            'Ticker': tickers,
            'Peso': pesos,
            'Precio_Actual': 0.0,
            'Valor_Total': 0.0,
            'Fecha_Actualización': datetime.now().strftime("%Y-%m-%d")
        })
        
        # Guardar
        ruta_archivo = self.ruta_cartera / f"{nombre_cartera}.xlsx"
        df.to_excel(ruta_archivo, index=False, sheet_name="Cartera")
        
        # Formatear Excel
        self._formatear_excel(ruta_archivo)
        
        print(f"✓ Cartera creada: {ruta_archivo}")
        return ruta_archivo
    
    def cargar_cartera(self, nombre_cartera: str) -> pd.DataFrame:
        """
        Carga una cartera Excel existente.
        
        Args:
            nombre_cartera: Nombre del archivo (con o sin .xlsx)
        
        Returns:
            DataFrame con los datos de la cartera
        """
        if not nombre_cartera.endswith('.xlsx'):
            nombre_cartera += '.xlsx'
        
        ruta_archivo = self.ruta_cartera / nombre_cartera
        
        if not ruta_archivo.exists():
            raise FileNotFoundError(f"Cartera no encontrada: {ruta_archivo}")
        
        self.datos_cartera = pd.read_excel(ruta_archivo)
        self.cartera_actual = nombre_cartera
        print(f"✓ Cartera cargada: {nombre_cartera}")
        return self.datos_cartera
    
    def obtener_tickers(self, cartera: pd.DataFrame = None) -> List[str]:
        """
        Obtiene lista de tickers de la cartera.
        
        Args:
            cartera: DataFrame (si None, usa self.datos_cartera)
        
        Returns:
            Lista de tickers
        """
        if cartera is None:
            cartera = self.datos_cartera
        
        if cartera is None:
            raise ValueError("No hay cartera cargada")
        
        return cartera['Ticker'].tolist()
    
    def actualizar_precios(self, precios: Dict[str, float]) -> pd.DataFrame:
        """
        Actualiza los precios de los tickers en la cartera.
        
        Args:
            precios: Diccionario {ticker: precio}
        
        Returns:
            DataFrame actualizado
        """
        if self.datos_cartera is None:
            raise ValueError("No hay cartera cargada")
        
        for ticker, precio in precios.items():
            if ticker in self.datos_cartera['Ticker'].values:
                idx = self.datos_cartera[self.datos_cartera['Ticker'] == ticker].index[0]
                self.datos_cartera.loc[idx, 'Precio_Actual'] = precio
        
        # Recalcular valores totales
        self.datos_cartera['Valor_Total'] = (
            self.datos_cartera['Peso'] * 
            self.datos_cartera['Precio_Actual']
        )
        
        self.datos_cartera['Fecha_Actualización'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        print(f"✓ Precios actualizados para {len(precios)} tickers")
        return self.datos_cartera
    
    def guardar_cartera(self, cartera: pd.DataFrame = None) -> None:
        """
        Guarda la cartera en Excel.
        
        Args:
            cartera: DataFrame (si None, usa self.datos_cartera)
        """
        if cartera is None:
            cartera = self.datos_cartera
        
        if cartera is None or self.cartera_actual is None:
            raise ValueError("No hay cartera cargada para guardar")
        
        ruta_archivo = self.ruta_cartera / self.cartera_actual
        cartera.to_excel(ruta_archivo, index=False, sheet_name="Cartera")
        self._formatear_excel(ruta_archivo)
        print(f"✓ Cartera guardada: {ruta_archivo}")
    
    def obtener_resumen_cartera(self) -> Dict:
        """Retorna resumen estadístico de la cartera."""
        if self.datos_cartera is None:
            raise ValueError("No hay cartera cargada")
        
        return {
            "num_tickers": len(self.datos_cartera),
            "peso_total": self.datos_cartera['Peso'].sum(),
            "valor_total": self.datos_cartera['Valor_Total'].sum(),
            "ultima_actualización": self.datos_cartera['Fecha_Actualización'].iloc[-1]
        }
    
    def _formatear_excel(self, ruta_archivo: Path) -> None:
        """Formatea el archivo Excel con estilos."""
        wb = openpyxl.load_workbook(ruta_archivo)
        ws = wb.active
        
        # Estilos
        header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
        header_font = Font(bold=True, color="FFFFFF")
        
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center")
        
        # Ancho de columnas
        ws.column_dimensions['A'].width = 12
        ws.column_dimensions['B'].width = 12
        ws.column_dimensions['C'].width = 15
        ws.column_dimensions['D'].width = 15
        ws.column_dimensions['E'].width = 20
        
        wb.save(ruta_archivo)
    
    def __repr__(self) -> str:
        return f"GestorCartera(cartera_actual='{self.cartera_actual}')"


## Clase 3 — DescargadorYahoo
Descargar tickers, guardar CSVs, leer CSVs, caché. Unas 150 líneas.


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from datetime import datetime, timedelta
import yfinance as yf
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

class DescargadorYahoo:
    """
    Descarga datos de tickers desde Yahoo Finance.
    Gestiona caché local, CSVs, y manejo de errores.
    """
    
    def __init__(self, gestor_rutas, usar_cache: bool = True):
        """
        Inicializa el descargador de Yahoo Finance.
        
        Args:
            gestor_rutas: Instancia de GestorRutas
            usar_cache: Si True, usa datos en caché cuando sea posible
        """
        self.gestor_rutas = gestor_rutas
        self.ruta_datos_csv = gestor_rutas.obtener_ruta("datos_csv")
        self.usar_cache = usar_cache
        self.cache_memoria = {}  # Caché en memoria
        self.fecha_cache = {}    # Fecha de última actualización por ticker
        self.duracion_cache = timedelta(hours=1)  # Duración del caché en memoria
    
    def descargar_datos(self, ticker: str, periodo: str = "1y", 
                       intervalo: str = "1d", forzar_descarga: bool = False) -> pd.DataFrame:
        """
        Descarga datos históricos de un ticker desde Yahoo Finance.
        
        Args:
            ticker: Símbolo del ticker (ej: 'AAPL')
            periodo: Período de datos ('1d', '5d', '1mo', '3mo', '6mo', '1y', '2y', '5y', '10y', 'max')
            intervalo: Intervalo de datos ('1m', '5m', '15m', '30m', '60m', '1d', '1wk', '1mo')
            forzar_descarga: Si True, descarga sin usar caché
        
        Returns:
            DataFrame con los datos históricos
        
        Raises:
            ValueError: Si el ticker es inválido o no se puede descargar
        """
        ticker = ticker.upper()
        
        # Verificar caché en memoria
        if not forzar_descarga and self.usar_cache:
            if self._verificar_cache_memoria(ticker):
                print(f"�� Usando caché en memoria para {ticker}")
                return self.cache_memoria[ticker].copy()
        
        try:
            print(f"📥 Descargando {ticker} desde Yahoo Finance...")
            
            # Descargar datos
            datos = yf.download(
                ticker, 
                period=periodo, 
                interval=intervalo, 
                progress=False,
                repair=True
            )
            
            if datos.empty:
                raise ValueError(f"No se obtuvieron datos para {ticker}")
            
            # Limpiar datos
            datos = self._limpiar_datos(datos)
            
            # Guardar en caché de memoria
            self.cache_memoria[ticker] = datos.copy()
            self.fecha_cache[ticker] = datetime.now()
            
            print(f"✓ {ticker}: {len(datos)} registros descargados")
            return datos
        
        except Exception as e:
            print(f"❌ Error descargando {ticker}: {e}")
            raise
    
    def descargar_multiples(self, tickers: List[str], periodo: str = "1y", 
                           intervalo: str = "1d") -> Dict[str, pd.DataFrame]:
        """
        Descarga datos para múltiples tickers.
        
        Args:
            tickers: Lista de tickers
            periodo: Período de datos
            intervalo: Intervalo de datos
        
        Returns:
            Diccionario {ticker: DataFrame}
        """
        datos_multiples = {}
        exitosos = 0
        fallidos = 0
        
        print(f"📥 Descargando {len(tickers)} tickers...")
        print("=" * 60)
        
        for ticker in tickers:
            try:
                datos = self.descargar_datos(ticker, periodo, intervalo)
                datos_multiples[ticker] = datos
                exitosos += 1
            except Exception as e:
                print(f"⚠️  {ticker}: {str(e)[:50]}")
                fallidos += 1
        
        print("=" * 60)
        print(f"✓ {exitosos} exitosos, ❌ {fallidos} fallidos")
        return datos_multiples
    
    def descargar_informacion_ticker(self, ticker: str) -> Dict:
        """
        Descarga información general del ticker.
        
        Args:
            ticker: Símbolo del ticker
        
        Returns:
            Diccionario con información del ticker
        """
        ticker = ticker.upper()
        
        try:
            info = yf.Ticker(ticker).info
            
            info_limpia = {
                'ticker': ticker,
                'nombre': info.get('longName', 'N/A'),
                'sector': info.get('sector', 'N/A'),
                'industria': info.get('industry', 'N/A'),
                'pais': info.get('country', 'N/A'),
                'precio_actual': info.get('currentPrice', 0),
                'precio_52_semanas_max': info.get('fiftyTwoWeekHigh', 0),
                'precio_52_semanas_min': info.get('fiftyTwoWeekLow', 0),
                'capitalizacion': info.get('marketCap', 0),
                'volumen_promedio': info.get('averageVolume', 0),
                'pe_ratio': info.get('trailingPE', 0),
                'dividend_yield': info.get('dividendYield', 0),
                'descripcion': info.get('longBusinessSummary', 'N/A')
            }
            
            print(f"✓ Información descargada para {ticker}")
            return info_limpia
        
        except Exception as e:
            print(f"❌ Error obteniendo información de {ticker}: {e}")
            return {}
    
    def guardar_csv(self, ticker: str, datos: pd.DataFrame, 
                   subcarpeta: str = None) -> Path:
        """
        Guarda datos en un archivo CSV.
        
        Args:
            ticker: Símbolo del ticker
            datos: DataFrame con los datos
            subcarpeta: Subcarpeta dentro de Datos_csv (opcional)
        
        Returns:
            Path: Ruta del archivo guardado
        """
        if subcarpeta:
            ruta_destino = self.ruta_datos_csv / subcarpeta
            ruta_destino.mkdir(parents=True, exist_ok=True)
        else:
            ruta_destino = self.ruta_datos_csv
        
        ruta_destino.mkdir(parents=True, exist_ok=True)
        ruta_archivo = ruta_destino / f"{ticker}.csv"
        
        try:
            datos.to_csv(ruta_archivo)
            print(f"✓ CSV guardado: {ruta_archivo}")
            return ruta_archivo
        except Exception as e:
            print(f"❌ Error guardando CSV: {e}")
            raise
    
    def guardar_multiples_csv(self, datos_multiples: Dict[str, pd.DataFrame], 
                             subcarpeta: str = None) -> List[Path]:
        """
        Guarda múltiples DataFrames como CSVs.
        
        Args:
            datos_multiples: Diccionario {ticker: DataFrame}
            subcarpeta: Subcarpeta dentro de Datos_csv
        
        Returns:
            Lista de rutas guardadas
        """
        rutas_guardadas = []
        
        for ticker, datos in datos_multiples.items():
            try:
                ruta = self.guardar_csv(ticker, datos, subcarpeta)
                rutas_guardadas.append(ruta)
            except Exception as e:
                print(f"⚠️  Error guardando {ticker}: {e}")
        
        print(f"\n✓ {len(rutas_guardadas)} archivos CSV guardados")
        return rutas_guardadas
    
    def leer_csv(self, ticker: str, subcarpeta: str = None) -> pd.DataFrame:
        """
        Lee datos de un archivo CSV.
        
        Args:
            ticker: Símbolo del ticker
            subcarpeta: Subcarpeta dentro de Datos_csv
        
        Returns:
            DataFrame con los datos
        
        Raises:
            FileNotFoundError: Si el archivo no existe
        """
        ticker = ticker.upper()
        
        if subcarpeta:
            ruta_archivo = self.ruta_datos_csv / subcarpeta / f"{ticker}.csv"
        else:
            ruta_archivo = self.ruta_datos_csv / f"{ticker}.csv"
        
        if not ruta_archivo.exists():
            raise FileNotFoundError(f"Archivo no encontrado: {ruta_archivo}")
        
        try:
            datos = pd.read_csv(ruta_archivo, index_col=0, parse_dates=True)
            print(f"✓ CSV leído: {ticker} ({len(datos)} registros)")
            return datos
        except Exception as e:
            print(f"❌ Error leyendo CSV: {e}")
            raise
    
    def leer_multiples_csv(self, tickers: List[str], subcarpeta: str = None) -> Dict[str, pd.DataFrame]:
        """
        Lee múltiples archivos CSV.
        
        Args:
            tickers: Lista de tickers
            subcarpeta: Subcarpeta dentro de Datos_csv
        
        Returns:
            Diccionario {ticker: DataFrame}
        """
        datos_multiples = {}
        exitosos = 0
        fallidos = 0
        
        print(f"📖 Leyendo {len(tickers)} CSVs...")
        
        for ticker in tickers:
            try:
                datos = self.leer_csv(ticker, subcarpeta)
                datos_multiples[ticker] = datos
                exitosos += 1
            except FileNotFoundError:
                print(f"⚠️  {ticker}: CSV no encontrado")
                fallidos += 1
            except Exception as e:
                print(f"⚠️  {ticker}: {str(e)[:50]}")
                fallidos += 1
        
        print(f"✓ {exitosos} exitosos, ❌ {fallidos} fallidos")
        return datos_multiples
    
    def obtener_precio_actual(self, ticker: str) -> float:
        """
        Obtiene el precio actual de un ticker.
        
        Args:
            ticker: Símbolo del ticker
        
        Returns:
            Precio actual en float
        """
        ticker = ticker.upper()
        
        try:
            datos = yf.download(ticker, period="1d", progress=False)
            if datos.empty:
                raise ValueError(f"No se obtuvieron datos para {ticker}")
            
            precio = datos['Close'].iloc[-1]
            print(f"✓ {ticker}: ${precio:.2f}")
            return float(precio)
        
        except Exception as e:
            print(f"❌ Error obteniendo precio de {ticker}: {e}")
            raise
    
    def obtener_precios_actuales(self, tickers: List[str]) -> Dict[str, float]:
        """
        Obtiene precios actuales para múltiples tickers.
        
        Args:
            tickers: Lista de tickers
        
        Returns:
            Diccionario {ticker: precio}
        """
        precios = {}
        
        print(f"💰 Obteniendo precios actuales de {len(tickers)} tickers...")
        
        for ticker in tickers:
            try:
                precio = self.obtener_precio_actual(ticker)
                precios[ticker] = precio
            except Exception as e:
                print(f"⚠️  {ticker}: No se pudo obtener precio")
        
        return precios
    
    def limpiar_cache_memoria(self, ticker: str = None) -> None:
        """
        Limpia el caché en memoria.
        
        Args:
            ticker: Ticker específico (si None, limpia todo)
        """
        if ticker is None:
            self.cache_memoria.clear()
            self.fecha_cache.clear()
            print("✓ Caché en memoria limpiado")
        else:
            ticker = ticker.upper()
            if ticker in self.cache_memoria:
                del self.cache_memoria[ticker]
                del self.fecha_cache[ticker]
                print(f"✓ Caché de {ticker} eliminado")
    
    def _verificar_cache_memoria(self, ticker: str) -> bool:
        """
        Verifica si hay datos válidos en caché de memoria.
        
        Args:
            ticker: Símbolo del ticker
        
        Returns:
            True si hay caché válido, False en caso contrario
        """
        if ticker not in self.cache_memoria:
            return False
        
        if ticker not in self.fecha_cache:
            return False
        
        edad_cache = datetime.now() - self.fecha_cache[ticker]
        return edad_cache < self.duracion_cache
    
    def _limpiar_datos(self, datos: pd.DataFrame) -> pd.DataFrame:
        """
        Limpia y formatea los datos descargados.
        
        Args:
            datos: DataFrame con datos crudos
        
        Returns:
            DataFrame limpiado
        """
        # Remover duplicados
        datos = datos[~datos.index.duplicated(keep='first')]
        
        # Ordenar por fecha
        datos = datos.sort_index()
        
        # Rellenar valores faltantes
        datos = datos.fillna(method='ffill').fillna(method='bfill')
        
        # Resetear nombres si es necesario
        if isinstance(datos.columns, pd.MultiIndex):
            datos.columns = [col[0] for col in datos.columns]
        
        return datos
    
    def exportar_resumen_descargas(self) -> str:
        """
        Genera un resumen de descargas realizadas.
        
        Returns:
            String con el resumen formateado
        """
        resumen = f"""
╔════════════════════════════════════════════════════════════╗
║              RESUMEN DE DESCARGAS YAHOO FINANCE             ║
╚════════════════════════════════════════════════════════════╝

Caché en memoria (activos):
  Tickers: {len(self.cache_memoria)}
  
"""
        
        if self.cache_memoria:
            resumen += "┌────────────────────────────────────────────────────────────┐\n"
            resumen += "│ Ticker │ Registros │ Fecha de actualización         │\n"
            resumen += "├────────┼───────────┼────────────────────────────────┤\n"
            
            for ticker in sorted(self.cache_memoria.keys()):
                registros = len(self.cache_memoria[ticker])
                fecha = self.fecha_cache[ticker].strftime("%Y-%m-%d %H:%M:%S")
                resumen += f"│ {ticker:6} │ {registros:9} │ {fecha:30} │\n"
            
            resumen += "└────────┴───────────┴────────────────────────────────────┘\n"
        
        # CSVs en disco
        csvs = list(self.ruta_datos_csv.glob("*.csv"))
        resumen += f"\nArchivos CSV en disco: {len(csvs)}\n"
        
        if csvs:
            resumen += "  Primeros 10 archivos:\n"
            for csv in csvs[:10]:
                tamaño = csv.stat().st_size / 1024  # KB
                resumen += f"    • {csv.name} ({tamaño:.1f} KB)\n"
        
        resumen += "\nConfiguración del descargador:\n"
        resumen += f"  Usar caché: {self.usar_cache}\n"
        resumen += f"  Duración caché: {self.duracion_cache}\n"
        
        return resumen
    
    def obtener_estadisticas_csv(self, ticker: str, subcarpeta: str = None) -> Dict:
        """
        Obtiene estadísticas de un archivo CSV.
        
        Args:
            ticker: Símbolo del ticker
            subcarpeta: Subcarpeta dentro de Datos_csv
        
        Returns:
            Diccionario con estadísticas
        """
        try:
            datos = self.leer_csv(ticker, subcarpeta)
            
            stats = {
                'ticker': ticker,
                'registros': len(datos),
                'fecha_inicio': datos.index[0].strftime("%Y-%m-%d") if len(datos) > 0 else None,
                'fecha_fin': datos.index[-1].strftime("%Y-%m-%d") if len(datos) > 0 else None,
                'precio_min': datos['Close'].min() if 'Close' in datos.columns else None,
                'precio_max': datos['Close'].max() if 'Close' in datos.columns else None,
                'precio_promedio': datos['Close'].mean() if 'Close' in datos.columns else None,
                'volumen_promedio': datos['Volume'].mean() if 'Volume' in datos.columns else None,
                'columnas': list(datos.columns)
            }
            
            return stats
        except Exception as e:
            print(f"❌ Error obteniendo estadísticas: {e}")
            return {}
    
    def crear_subcarpe_historico(self, nombre: str = None) -> Path:
        """
        Crea una subcarpeta para guardar datos históricos.
        
        Args:
            nombre: Nombre de la subcarpeta (default: fecha actual)
        
        Returns:
            Path: Ruta de la subcarpeta creada
        """
        if nombre is None:
            nombre = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        ruta_subcarpe = self.ruta_datos_csv / nombre
        ruta_subcarpe.mkdir(parents=True, exist_ok=True)
        print(f"✓ Subcarpeta creada: {ruta_subcarpe}")
        return ruta_subcarpe
    
    def __repr__(self) -> str:
        return f"DescargadorYahoo(cache_memoria={len(self.cache_memoria)}, usar_cache={self.usar_cache})"

## Clase 4 — AnalizadorMetricas
_metricas_csv, _diagnostico, Sharpe, Sortino, Omega, Skew, Kurtosis, Tail Ratio. Unas 250 líneas.


In [5]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

class AnalizadorMetricas:
    """
    Calcula métricas de rendimiento y riesgo para carteras y activos individuales.
    Incluye Sharpe, Sortino, Omega, Skew, Kurtosis, Tail Ratio y más.
    """
    
    def __init__(self, tasa_libre_riesgo: float = 0.02):
        """
        Inicializa el analizador de métricas.
        
        Args:
            tasa_libre_riesgo: Tasa libre de riesgo anual (default: 2%)
        """
        self.tasa_libre_riesgo = tasa_libre_riesgo
        self.retornos_cache = {}
        self.metricas_cache = {}
    
    def calcular_retornos_diarios(self, datos: pd.DataFrame, 
                                  columna: str = 'Close') -> pd.Series:
        """
        Calcula los retornos diarios a partir de precios.
        
        Args:
            datos: DataFrame con datos históricos
            columna: Nombre de la columna de precios (default: 'Close')
        
        Returns:
            Series con retornos diarios (logarítmicos)
        """
        if columna not in datos.columns:
            raise ValueError(f"Columna '{columna}' no encontrada")
        
        precios = datos[columna]
        retornos = np.log(precios / precios.shift(1)).dropna()
        
        return retornos
    
    def calcular_retornos_anualizados(self, retornos: pd.Series, 
                                      dias_trading: int = 252) -> float:
        """
        Calcula el retorno anualizado.
        
        Args:
            retornos: Series de retornos diarios
            dias_trading: Días de trading por año (default: 252)
        
        Returns:
            Retorno anualizado en decimal
        """
        retorno_compuesto = (1 + retornos).prod() - 1
        años = len(retornos) / dias_trading
        return (1 + retorno_compuesto) ** (1 / años) - 1
    
    def calcular_volatilidad_anualizada(self, retornos: pd.Series, 
                                       dias_trading: int = 252) -> float:
        """
        Calcula la volatilidad anualizada.
        
        Args:
            retornos: Series de retornos diarios
            dias_trading: Días de trading por año (default: 252)
        
        Returns:
            Volatilidad anualizada en decimal
        """
        volatilidad_diaria = retornos.std()
        return volatilidad_diaria * np.sqrt(dias_trading)
    
    def calcular_sharpe_ratio(self, retornos: pd.Series, 
                             dias_trading: int = 252) -> float:
        """
        Calcula el Ratio de Sharpe.
        
        Formula: (Retorno anualizado - Tasa libre de riesgo) / Volatilidad anualizada
        
        Args:
            retornos: Series de retornos diarios
            dias_trading: Días de trading por año (default: 252)
        
        Returns:
            Ratio de Sharpe
        """
        if len(retornos) < 2:
            return np.nan
        
        retorno_anualizado = self.calcular_retornos_anualizados(retornos, dias_trading)
        volatilidad = self.calcular_volatilidad_anualizada(retornos, dias_trading)
        
        if volatilidad == 0:
            return np.nan
        
        sharpe = (retorno_anualizado - self.tasa_libre_riesgo) / volatilidad
        return sharpe
    
    def calcular_sortino_ratio(self, retornos: pd.Series, 
                              dias_trading: int = 252) -> float:
        """
        Calcula el Ratio de Sortino.
        
        Similar a Sharpe pero solo considera la volatilidad a la baja (downside).
        
        Args:
            retornos: Series de retornos diarios
            dias_trading: Días de trading por año (default: 252)
        
        Returns:
            Ratio de Sortino
        """
        if len(retornos) < 2:
            return np.nan
        
        retorno_anualizado = self.calcular_retornos_anualizados(retornos, dias_trading)
        
        # Volatilidad a la baja (solo retornos negativos)
        retornos_negativos = retornos[retornos < 0]
        if len(retornos_negativos) == 0:
            downside_volatilidad = 0
        else:
            downside_volatilidad = retornos_negativos.std() * np.sqrt(dias_trading)
        
        if downside_volatilidad == 0:
            return np.nan
        
        sortino = (retorno_anualizado - self.tasa_libre_riesgo) / downside_volatilidad
        return sortino
    
    def calcular_omega_ratio(self, retornos: pd.Series, 
                            target_retorno: float = 0.0) -> float:
        """
        Calcula el Ratio Omega.
        
        Ratio de probabilidad ponderada de ganancias vs pérdidas.
        
        Args:
            retornos: Series de retornos diarios
            target_retorno: Retorno objetivo (default: 0.0)
        
        Returns:
            Ratio Omega
        """
        if len(retornos) < 2:
            return np.nan
        
        ganancias = retornos[retornos > target_retorno] - target_retorno
        perdidas = target_retorno - retornos[retornos < target_retorno]
        
        ganancia_total = ganancias.sum()
        perdida_total = perdidas.sum()
        
        if perdida_total == 0:
            return np.inf if ganancia_total > 0 else 1.0
        
        omega = ganancia_total / perdida_total
        return omega
    
    def calcular_skewness(self, retornos: pd.Series) -> float:
        """
        Calcula la asimetría (Skewness) de los retornos.
        
        Negativo: cola izquierda (más pérdidas extremas)
        Positivo: cola derecha (más ganancias extremas)
        
        Args:
            retornos: Series de retornos diarios
        
        Returns:
            Coeficiente de asimetría
        """
        if len(retornos) < 3:
            return np.nan
        
        return stats.skew(retornos)
    
    def calcular_kurtosis(self, retornos: pd.Series) -> float:
        """
        Calcula la curtosis de los retornos.
        
        Exceso de curtosis: > 0 (colas pesadas), < 0 (colas ligeras)
        
        Args:
            retornos: Series de retornos diarios
        
        Returns:
            Curtosis (exceso)
        """
        if len(retornos) < 4:
            return np.nan
        
        return stats.kurtosis(retornos)
    
    def calcular_tail_ratio(self, retornos: pd.Series, 
                           percentil: float = 0.05) -> float:
        """
        Calcula el Tail Ratio (relación de colas).
        
        Ratio entre pérdidas extremas y ganancias extremas.
        > 1 indica asimetría hacia abajo
        
        Args:
            retornos: Series de retornos diarios
            percentil: Percentil para definir la cola (default: 5%)
        
        Returns:
            Tail Ratio
        """
        if len(retornos) < 20:
            return np.nan
        
        cola_negativa = abs(np.percentile(retornos, percentil * 100))
        cola_positiva = np.percentile(retornos, (1 - percentil) * 100)
        
        if cola_positiva == 0:
            return np.nan
        
        tail_ratio = cola_negativa / cola_positiva
        return tail_ratio
    
    def calcular_drawdown_maximo(self, datos: pd.DataFrame, 
                                columna: str = 'Close') -> float:
        """
        Calcula el drawdown máximo.
        
        Pérdida máxima desde el máximo histórico.
        
        Args:
            datos: DataFrame con datos históricos
            columna: Nombre de la columna de precios
        
        Returns:
            Drawdown máximo en decimal (negativo)
        """
        if columna not in datos.columns:
            raise ValueError(f"Columna '{columna}' no encontrada")
        
        precios = datos[columna]
        valor_maximo = precios.expanding().max()
        drawdown = (precios - valor_maximo) / valor_maximo
        
        return drawdown.min()
    
    def calcular_calmar_ratio(self, retornos: pd.Series, 
                             datos: pd.DataFrame,
                             columna: str = 'Close',
                             dias_trading: int = 252) -> float:
        """
        Calcula el Ratio de Calmar.
        
        Retorno anualizado / |Drawdown máximo|
        
        Args:
            retornos: Series de retornos diarios
            datos: DataFrame con datos históricos
            columna: Nombre de la columna de precios
            dias_trading: Días de trading por año
        
        Returns:
            Ratio de Calmar
        """
        if len(retornos) < 2:
            return np.nan
        
        retorno_anualizado = self.calcular_retornos_anualizados(retornos, dias_trading)
        drawdown_max = self.calcular_drawdown_maximo(datos, columna)
        
        if drawdown_max == 0:
            return np.nan
        
        calmar = retorno_anualizado / abs(drawdown_max)
        return calmar
    
    def calcular_var(self, retornos: pd.Series, 
                    nivel_confianza: float = 0.95) -> float:
        """
        Calcula el Value at Risk (VaR).
        
        Pérdida potencial en el peor caso con nivel de confianza dado.
        
        Args:
            retornos: Series de retornos diarios
            nivel_confianza: Nivel de confianza (default: 95%)
        
        Returns:
            VaR en decimal (negativo)
        """
        if len(retornos) < 2:
            return np.nan
        
        var = np.percentile(retornos, (1 - nivel_confianza) * 100)
        return var
    
    def calcular_cvar(self, retornos: pd.Series, 
                     nivel_confianza: float = 0.95) -> float:
        """
        Calcula el Conditional Value at Risk (CVaR) / Expected Shortfall.
        
        Pérdida promedio cuando el retorno es peor que el VaR.
        
        Args:
            retornos: Series de retornos diarios
            nivel_confianza: Nivel de confianza (default: 95%)
        
        Returns:
            CVaR en decimal (negativo)
        """
        if len(retornos) < 2:
            return np.nan
        
        var = self.calcular_var(retornos, nivel_confianza)
        cvar = retornos[retornos <= var].mean()
        
        return cvar
    
    def calcular_retorno_sobre_volatilidad(self, retornos: pd.Series,
                                          dias_trading: int = 252) -> float:
        """
        Calcula el Retorno sobre Volatilidad.
        
        Retorno anualizado / Volatilidad anualizada
        
        Args:
            retornos: Series de retornos diarios
            dias_trading: Días de trading por año
        
        Returns:
            Ratio Retorno/Volatilidad
        """
        if len(retornos) < 2:
            return np.nan
        
        retorno_anualizado = self.calcular_retornos_anualizados(retornos, dias_trading)
        volatilidad = self.calcular_volatilidad_anualizada(retornos, dias_trading)
        
        if volatilidad == 0:
            return np.nan
        
        return retorno_anualizado / volatilidad
    
    def calcular_metricas_csv(self, datos: pd.DataFrame, 
                             nombre_activo: str = None,
                             columna: str = 'Close') -> Dict:
        """
        Calcula todas las métricas para un CSV de datos.
        
        Args:
            datos: DataFrame con datos históricos
            nombre_activo: Nombre del activo (para referencia)
            columna: Nombre de la columna de precios
        
        Returns:
            Diccionario con todas las métricas
        """
        try:
            retornos = self.calcular_retornos_diarios(datos, columna)
            
            metricas = {
                'nombre': nombre_activo or 'Activo',
                'registros': len(datos),
                'fecha_inicio': datos.index[0].strftime("%Y-%m-%d") if len(datos) > 0 else None,
                'fecha_fin': datos.index[-1].strftime("%Y-%m-%d") if len(datos) > 0 else None,
                'precio_inicial': datos[columna].iloc[0],
                'precio_final': datos[columna].iloc[-1],
                'precio_min': datos[columna].min(),
                'precio_max': datos[columna].max(),
                'retorno_total': (datos[columna].iloc[-1] / datos[columna].iloc[0]) - 1,
                'retorno_anualizado': self.calcular_retornos_anualizados(retornos),
                'volatilidad_anualizada': self.calcular_volatilidad_anualizada(retornos),
                'sharpe_ratio': self.calcular_sharpe_ratio(retornos),
                'sortino_ratio': self.calcular_sortino_ratio(retornos),
                'omega_ratio': self.calcular_omega_ratio(retornos),
                'skewness': self.calcular_skewness(retornos),
                'kurtosis': self.calcular_kurtosis(retornos),
                'tail_ratio': self.calcular_tail_ratio(retornos),
                'drawdown_maximo': self.calcular_drawdown_maximo(datos, columna),
                'calmar_ratio': self.calcular_calmar_ratio(retornos, datos, columna),
                'var_95': self.calcular_var(retornos, 0.95),
                'cvar_95': self.calcular_cvar(retornos, 0.95),
                'retorno_volatilidad': self.calcular_retorno_sobre_volatilidad(retornos),
                'dias_positivos': (retornos > 0).sum(),
                'dias_negativos': (retornos < 0).sum(),
                'ratio_ganancias_perdidas': (retornos > 0).sum() / (retornos < 0).sum() if (retornos < 0).sum() > 0 else np.inf
            }
            
            return metricas
        
        except Exception as e:
            print(f"❌ Error calculando métricas: {e}")
            return {}
    
    def calcular_diagnostico(self, metricas: Dict) -> Dict:
        """
        Análisis diagnóstico basado en las métricas calculadas.
        
        Args:
            metricas: Diccionario de métricas
        
        Returns:
            Diccionario con diagnóstico
        """
        diagnostico = {
            'nombre': metricas.get('nombre'),
            'evaluaciones': [],
            'puntuacion_general': 0
        }
        
        # Sharpe Ratio
        sharpe = metricas.get('sharpe_ratio', 0)
        if sharpe > 1:
            diagnostico['evaluaciones'].append("✓ Excelente Sharpe Ratio (> 1.0)")
            diagnostico['puntuacion_general'] += 3
        elif sharpe > 0.5:
            diagnostico['evaluaciones'].append("○ Buen Sharpe Ratio (0.5 - 1.0)")
            diagnostico['puntuacion_general'] += 2
        elif sharpe > 0:
            diagnostico['evaluaciones'].append("△ Sharpe Ratio moderado (0 - 0.5)")
            diagnostico['puntuacion_general'] += 1
        else:
            diagnostico['evaluaciones'].append("✗ Sharpe Ratio negativo (< 0)")
        
        # Sortino Ratio
        sortino = metricas.get('sortino_ratio', 0)
        if sortino > 1.5:
            diagnostico['evaluaciones'].append("✓ Excelente Sortino Ratio (> 1.5)")
            diagnostico['puntuacion_general'] += 3
        elif sortino > 0.5:
            diagnostico['evaluaciones'].append("○ Buen Sortino Ratio")
            diagnostico['puntuacion_general'] += 1
        
        # Skewness
        skew = metricas.get('skewness', 0)
        if skew > 0.5:
            diagnostico['evaluaciones'].append("✓ Skewness positivo (cola de ganancias)")
            diagnostico['puntuacion_general'] += 2
        elif skew < -0.5:
            diagnostico['evaluaciones'].append("✗ Skewness negativo (cola de pérdidas)")
        
        # Kurtosis
        kurt = metricas.get('kurtosis', 0)
        if kurt > 2:
            diagnostico['evaluaciones'].append("⚠ Kurtosis alto (colas pesadas, riesgo de eventos extremos)")
        
        # Drawdown
        dd = metricas.get('drawdown_maximo', 0)
        if dd > -0.1:
            diagnostico['evaluaciones'].append("✓ Drawdown contenido (> -10%)")
            diagnostico['puntuacion_general'] += 2
        elif dd > -0.25:
            diagnostico['evaluaciones'].append("○ Drawdown moderado (-10% a -25%)")
            diagnostico['puntuacion_general'] += 1
        else:
            diagnostico['evaluaciones'].append("✗ Drawdown severo (< -25%)")
        
        # Retorno
        ret = metricas.get('retorno_total', 0)
        if ret > 0.2:
            diagnostico['evaluaciones'].append("✓ Retorno positivo (> 20%)")
            diagnostico['puntuacion_general'] += 3
        elif ret > 0:
            diagnostico['evaluaciones'].append("○ Retorno moderado positivo")
            diagnostico['puntuacion_general'] += 1
        else:
            diagnostico['evaluaciones'].append("✗ Retorno negativo")
        
        return diagnostico
    
    def exportar_resumen_metricas(self, metricas: Dict) -> str:
        """
        Genera un resumen formateado de las métricas.
        
        Args:
            metricas: Diccionario de métricas
        
        Returns:
            String con el resumen formateado
        """
        resumen = f"""
╔════════════════════════════════════════════════════════════╗
║              ANÁLISIS DE MÉTRICAS - {metricas.get('nombre', 'ACTIVO'):20} ║
╚════════════════════════════════════════════════════════════╝

INFORMACIÓN GENERAL:
  Período: {metricas.get('fecha_inicio')} a {metricas.get('fecha_fin')}
  Registros: {metricas.get('registros')}
  Precio inicial: ${metricas.get('precio_inicial', 0):.2f}
  Precio final: ${metricas.get('precio_final', 0):.2f}
  Rango: ${metricas.get('precio_min', 0):.2f} - ${metricas.get('precio_max', 0):.2f}

RETORNO Y VOLATILIDAD:
  Retorno total: {metricas.get('retorno_total', 0):.2%}
  Retorno anualizado: {metricas.get('retorno_anualizado', 0):.2%}
  Volatilidad anualizada: {metricas.get('volatilidad_anualizada', 0):.2%}
  Retorno/Volatilidad: {metricas.get('retorno_volatilidad', 0):.4f}

RATIOS DE RENDIMIENTO AJUSTADO:
  Sharpe Ratio: {metricas.get('sharpe_ratio', np.nan):.4f}
  Sortino Ratio: {metricas.get('sortino_ratio', np.nan):.4f}
  Omega Ratio: {metricas.get('omega_ratio', np.nan):.4f}
  Calmar Ratio: {metricas.get('calmar_ratio', np.nan):.4f}

ANÁLISIS DE DISTRIBUCIÓN:
  Skewness: {metricas.get('skewness', np.nan):.4f}
  Kurtosis: {metricas.get('kurtosis', np.nan):.4f}
  Tail Ratio: {metricas.get('tail_ratio', np.nan):.4f}

RIESGO:
  Drawdown máximo: {metricas.get('drawdown_maximo', 0):.2%}
  VaR (95%): {metricas.get('var_95', 0):.4f}
  CVaR (95%): {metricas.get('cvar_95', 0):.4f}

ACTIVIDAD:
  Días positivos: {metricas.get('dias_positivos')}
  Días negativos: {metricas.get('dias_negativos')}
  Ratio Ganancias/Pérdidas: {metricas.get('ratio_ganancias_perdidas', np.nan):.4f}
"""
        
        return resumen
    
    def __repr__(self) -> str:
        return f"AnalizadorMetricas(tasa_libre_riesgo={self.tasa_libre_riesgo:.2%})"

## Interpretación de las métricas:
| Métrica            | Rango        | Interpretación                     |
|--------------------|--------------|------------------------------------|
| **Sharpe Ratio**   | > 1.0        | Excelente                          |
|                    | 0.5 – 1.0    | Bueno                              |
|                    | 0 – 0.5      | Moderado                           |
|                    | < 0          | Malo                               |
| **Sortino Ratio**  | > 2.0        | Excelente                          |
|                    | 1.0 – 2.0    | Bueno                              |
|                    | < 1.0        | Malo                               |
| **Omega Ratio**    | > 1.5        | Muy favorable                      |
|                    | 1.0 – 1.5    | Favorable                          |
|                    | < 1.0        | Desfavorable                       |
| **Skewness**       | > 0          | Colas de ganancias                 |
|                    | < 0          | Colas de pérdidas                  |
| **Kurtosis**       | > 0          | Colas pesadas (riesgo)             |
|                    | < 0          | Colas ligeras                      |
| **Drawdown Máximo**| > –10%       | Bajo                               |
|                    | –10% a –25%  | Moderado                           |
|                    | < –25%       | Alto                               |


## Clase 5 — AnalizadorTecnico
SMA, Bollinger, MACD, ATR, RSI, diagnóstico técnico, gráficas técnicas. Unas 200 líneas.



In [6]:
import pandas as pd
import numpy as np
from typing import Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

class AnalizadorTecnico:
    """
    Calcula indicadores técnicos (SMA, Bollinger, MACD, ATR, RSI).
    Genera diagnósticos técnicos y gráficas profesionales.
    """
    
    def __init__(self):
        """Inicializa el analizador técnico."""
        self.indicadores_cache = {}
        sns.set_style("darkgrid")
    
    # ============================================
    # INDICADORES TÉCNICOS
    # ============================================
    
    def calcular_sma(self, datos: pd.DataFrame, periodo: int = 20, 
                    columna: str = 'Close') -> pd.Series:
        """
        Calcula el Promedio Móvil Simple (SMA).
        
        Args:
            datos: DataFrame con datos históricos
            periodo: Período del promedio (default: 20)
            columna: Columna a usar (default: 'Close')
        
        Returns:
            Series con SMA
        """
        if columna not in datos.columns:
            raise ValueError(f"Columna '{columna}' no encontrada")
        
        sma = datos[columna].rolling(window=periodo).mean()
        return sma
    
    def calcular_ema(self, datos: pd.DataFrame, periodo: int = 20,
                    columna: str = 'Close') -> pd.Series:
        """
        Calcula el Promedio Móvil Exponencial (EMA).
        
        Args:
            datos: DataFrame con datos históricos
            periodo: Período del promedio (default: 20)
            columna: Columna a usar (default: 'Close')
        
        Returns:
            Series con EMA
        """
        if columna not in datos.columns:
            raise ValueError(f"Columna '{columna}' no encontrada")
        
        ema = datos[columna].ewm(span=periodo, adjust=False).mean()
        return ema
    
    def calcular_bandas_bollinger(self, datos: pd.DataFrame, periodo: int = 20,
                                 desv_std: float = 2.0, columna: str = 'Close') -> Dict[str, pd.Series]:
        """
        Calcula las Bandas de Bollinger.
        
        Args:
            datos: DataFrame con datos históricos
            periodo: Período del promedio (default: 20)
            desv_std: Número de desviaciones estándar (default: 2.0)
            columna: Columna a usar (default: 'Close')
        
        Returns:
            Diccionario con 'banda_media', 'banda_superior', 'banda_inferior'
        """
        if columna not in datos.columns:
            raise ValueError(f"Columna '{columna}' no encontrada")
        
        banda_media = datos[columna].rolling(window=periodo).mean()
        desviacion = datos[columna].rolling(window=periodo).std()
        
        banda_superior = banda_media + (desv_std * desviacion)
        banda_inferior = banda_media - (desv_std * desviacion)
        
        return {
            'banda_media': banda_media,
            'banda_superior': banda_superior,
            'banda_inferior': banda_inferior,
            'banda_width': (banda_superior - banda_inferior) / banda_media * 100
        }
    
    def calcular_macd(self, datos: pd.DataFrame, fast: int = 12, slow: int = 26,
                     signal: int = 9, columna: str = 'Close') -> Dict[str, pd.Series]:
        """
        Calcula el MACD (Moving Average Convergence Divergence).
        
        Args:
            datos: DataFrame con datos históricos
            fast: Período rápido (default: 12)
            slow: Período lento (default: 26)
            signal: Período de señal (default: 9)
            columna: Columna a usar (default: 'Close')
        
        Returns:
            Diccionario con 'macd', 'signal', 'histogram'
        """
        if columna not in datos.columns:
            raise ValueError(f"Columna '{columna}' no encontrada")
        
        ema_fast = datos[columna].ewm(span=fast, adjust=False).mean()
        ema_slow = datos[columna].ewm(span=slow, adjust=False).mean()
        
        macd = ema_fast - ema_slow
        signal_line = macd.ewm(span=signal, adjust=False).mean()
        histogram = macd - signal_line
        
        return {
            'macd': macd,
            'signal': signal_line,
            'histogram': histogram
        }
    
    def calcular_rsi(self, datos: pd.DataFrame, periodo: int = 14,
                    columna: str = 'Close') -> pd.Series:
        """
        Calcula el Índice de Fuerza Relativa (RSI).
        
        Rango: 0-100
        > 70: Sobrecompra
        < 30: Sobreventa
        
        Args:
            datos: DataFrame con datos históricos
            periodo: Período del RSI (default: 14)
            columna: Columna a usar (default: 'Close')
        
        Returns:
            Series con RSI
        """
        if columna not in datos.columns:
            raise ValueError(f"Columna '{columna}' no encontrada")
        
        precios = datos[columna]
        cambios = precios.diff()
        
        ganancias = cambios.where(cambios > 0, 0)
        perdidas = -cambios.where(cambios < 0, 0)
        
        ganancia_promedio = ganancias.rolling(window=periodo).mean()
        perdida_promedio = perdidas.rolling(window=periodo).mean()
        
        rs = ganancia_promedio / perdida_promedio
        rsi = 100 - (100 / (1 + rs))
        
        return rsi
    
    def calcular_atr(self, datos: pd.DataFrame, periodo: int = 14) -> pd.Series:
        """
        Calcula el Rango Verdadero Promedio (ATR).
        
        Mide la volatilidad.
        
        Args:
            datos: DataFrame con datos históricos (requiere High, Low, Close)
            periodo: Período del ATR (default: 14)
        
        Returns:
            Series con ATR
        """
        columnas_requeridas = ['High', 'Low', 'Close']
        for col in columnas_requeridas:
            if col not in datos.columns:
                raise ValueError(f"Columna '{col}' no encontrada")
        
        high = datos['High']
        low = datos['Low']
        close = datos['Close']
        
        tr1 = high - low
        tr2 = abs(high - close.shift(1))
        tr3 = abs(low - close.shift(1))
        
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        atr = tr.rolling(window=periodo).mean()
        
        return atr
    
    def calcular_stochastic(self, datos: pd.DataFrame, periodo: int = 14,
                           suavizado: int = 3, columna: str = 'Close') -> Dict[str, pd.Series]:
        """
        Calcula el Oscilador Estocástico.
        
        Rango: 0-100
        > 80: Sobrecompra
        < 20: Sobreventa
        
        Args:
            datos: DataFrame con datos históricos (requiere High, Low, Close)
            periodo: Período del estocástico (default: 14)
            suavizado: Período de suavizado (default: 3)
            columna: Columna a usar (default: 'Close')
        
        Returns:
            Diccionario con '%K' y '%D'
        """
        if columna not in datos.columns or 'High' not in datos.columns or 'Low' not in datos.columns:
            raise ValueError("Se requieren columnas: High, Low, Close")
        
        low_min = datos['Low'].rolling(window=periodo).min()
        high_max = datos['High'].rolling(window=periodo).max()
        
        k_percent = 100 * ((datos[columna] - low_min) / (high_max - low_min))
        d_percent = k_percent.rolling(window=suavizado).mean()
        
        return {
            'k_percent': k_percent,
            'd_percent': d_percent
        }
    
    def calcular_williams_r(self, datos: pd.DataFrame, periodo: int = 14) -> pd.Series:
        """
        Calcula el Williams %R.
        
        Rango: -100 a 0
        > -20: Sobrecompra
        < -80: Sobreventa
        
        Args:
            datos: DataFrame con datos históricos (requiere High, Low, Close)
            periodo: Período (default: 14)
        
        Returns:
            Series con Williams %R
        """
        if 'High' not in datos.columns or 'Low' not in datos.columns or 'Close' not in datos.columns:
            raise ValueError("Se requieren columnas: High, Low, Close")
        
        high_max = datos['High'].rolling(window=periodo).max()
        low_min = datos['Low'].rolling(window=periodo).min()
        
        williams_r = -100 * ((high_max - datos['Close']) / (high_max - low_min))
        
        return williams_r
    
    def calcular_cci(self, datos: pd.DataFrame, periodo: int = 20) -> pd.Series:
        """
        Calcula el Índice de Canal de Commodities (CCI).
        
        Args:
            datos: DataFrame con datos históricos (requiere High, Low, Close)
            periodo: Período (default: 20)
        
        Returns:
            Series con CCI
        """
        if 'High' not in datos.columns or 'Low' not in datos.columns or 'Close' not in datos.columns:
            raise ValueError("Se requieren columnas: High, Low, Close")
        
        precio_tipico = (datos['High'] + datos['Low'] + datos['Close']) / 3
        sma_tipico = precio_tipico.rolling(window=periodo).mean()
        desv_media = precio_tipico.rolling(window=periodo).apply(
            lambda x: np.mean(np.abs(x - x.mean()))
        )
        
        cci = (precio_tipico - sma_tipico) / (0.015 * desv_media)
        
        return cci
    
    # ============================================
    # DIAGNÓSTICO TÉCNICO
    # ============================================
    
    def diagnostico_tecnico(self, datos: pd.DataFrame, 
                           nombre_activo: str = None) -> Dict:
        """
        Realiza un diagnóstico técnico completo.
        
        Args:
            datos: DataFrame con datos históricos
            nombre_activo: Nombre del activo
        
        Returns:
            Diccionario con diagnóstico técnico
        """
        try:
            diagnostico = {
                'nombre': nombre_activo or 'Activo',
                'fecha_analisis': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                'señales': [],
                'puntuacion': 0,
                'tendencia': 'Neutral'
            }
            
            precio_actual = datos['Close'].iloc[-1]
            precio_anterior = datos['Close'].iloc[-2] if len(datos) > 1 else precio_actual
            
            # SMA
            sma_20 = self.calcular_sma(datos, 20)
            sma_50 = self.calcular_sma(datos, 50)
            sma_actual_20 = sma_20.iloc[-1]
            sma_actual_50 = sma_50.iloc[-1]
            
            if precio_actual > sma_actual_20 > sma_actual_50:
                diagnostico['señales'].append("✓ SMA: Tendencia alcista fuerte")
                diagnostico['puntuacion'] += 2
                diagnostico['tendencia'] = 'Alcista'
            elif precio_actual < sma_actual_20 < sma_actual_50:
                diagnostico['señales'].append("✗ SMA: Tendencia bajista fuerte")
                diagnostico['tendencia'] = 'Bajista'
            
            # RSI
            rsi = self.calcular_rsi(datos)
            rsi_actual = rsi.iloc[-1]
            
            if rsi_actual > 70:
                diagnostico['señales'].append("⚠ RSI: Sobrecompra (> 70)")
            elif rsi_actual < 30:
                diagnostico['señales'].append("⚠ RSI: Sobreventa (< 30)")
                diagnostico['puntuacion'] += 1
            elif 40 < rsi_actual < 60:
                diagnostico['señales'].append("○ RSI: Neutral")
            
            # MACD
            macd_data = self.calcular_macd(datos)
            macd = macd_data['macd'].iloc[-1]
            signal = macd_data['signal'].iloc[-1]
            histogram = macd_data['histogram'].iloc[-1]
            
            if macd > signal and histogram > 0:
                diagnostico['señales'].append("✓ MACD: Cruce alcista")
                diagnostico['puntuacion'] += 1
            elif macd < signal and histogram < 0:
                diagnostico['señales'].append("✗ MACD: Cruce bajista")
            
            # Bandas de Bollinger
            bb = self.calcular_bandas_bollinger(datos)
            banda_sup = bb['banda_superior'].iloc[-1]
            banda_inf = bb['banda_inferior'].iloc[-1]
            
            if precio_actual > banda_sup:
                diagnostico['señales'].append("△ Bollinger: Precio por encima de banda superior (sobreextensión)")
            elif precio_actual < banda_inf:
                diagnostico['señales'].append("△ Bollinger: Precio por debajo de banda inferior")
            else:
                diagnostico['señales'].append("○ Bollinger: Precio dentro de bandas")
            
            return diagnostico
        
        except Exception as e:
            print(f"❌ Error en diagnóstico técnico: {e}")
            return {}
    
    # ============================================
    # GRÁFICAS
    # ============================================
    
    def grafica_precios_sma(self, datos: pd.DataFrame, nombre_activo: str = None,
                           periodos_sma: List[int] = None) -> None:
        """
        Grafica precios con SMAs superpuestos.
        
        Args:
            datos: DataFrame con datos históricos
            nombre_activo: Nombre del activo
            periodos_sma: Lista de períodos para SMA (default: [20, 50, 200])
        """
        if periodos_sma is None:
            periodos_sma = [20, 50, 200]
        
        plt.figure(figsize=(14, 6))
        
        # Precio
        plt.plot(datos.index, datos['Close'], label='Precio', linewidth=2, color='black')
        
        # SMAs
        colores = ['blue', 'orange', 'red']
        for periodo, color in zip(periodos_sma, colores):
            sma = self.calcular_sma(datos, periodo)
            plt.plot(datos.index, sma, label=f'SMA {periodo}', linewidth=1.5, color=color, alpha=0.7)
        
        plt.title(f'{nombre_activo or "Activo"} - Precio y Promedios Móviles', fontsize=14, fontweight='bold')
        plt.xlabel('Fecha', fontsize=12)
        plt.ylabel('Precio ($)', fontsize=12)
        plt.legend(loc='best')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    def grafica_bandas_bollinger(self, datos: pd.DataFrame, periodo: int = 20,
                                nombre_activo: str = None) -> None:
        """
        Grafica Bandas de Bollinger.
        
        Args:
            datos: DataFrame con datos históricos
            periodo: Período de las bandas
            nombre_activo: Nombre del activo
        """
        bb = self.calcular_bandas_bollinger(datos, periodo)
        
        plt.figure(figsize=(14, 6))
        
        # Bandas
        plt.fill_between(datos.index, bb['banda_superior'], bb['banda_inferior'],
                        alpha=0.2, color='blue', label='Bandas de Bollinger')
        plt.plot(datos.index, bb['banda_superior'], color='blue', linewidth=1, alpha=0.5)
        plt.plot(datos.index, bb['banda_inferior'], color='blue', linewidth=1, alpha=0.5)
        plt.plot(datos.index, bb['banda_media'], color='blue', linewidth=1.5, linestyle='--', label='Media')
        
        # Precio
        plt.plot(datos.index, datos['Close'], label='Precio', linewidth=2, color='black')
        
        plt.title(f'{nombre_activo or "Activo"} - Bandas de Bollinger', fontsize=14, fontweight='bold')
        plt.xlabel('Fecha', fontsize=12)
        plt.ylabel('Precio ($)', fontsize=12)
        plt.legend(loc='best')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    def grafica_macd(self, datos: pd.DataFrame, nombre_activo: str = None) -> None:
        """
        Grafica el MACD.
        
        Args:
            datos: DataFrame con datos históricos
            nombre_activo: Nombre del activo
        """
        macd_data = self.calcular_macd(datos)
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
        
        # Precio
        ax1.plot(datos.index, datos['Close'], label='Precio', linewidth=2, color='black')
        ax1.set_title(f'{nombre_activo or "Activo"} - Precio', fontsize=12, fontweight='bold')
        ax1.set_ylabel('Precio ($)', fontsize=11)
        ax1.legend(loc='best')
        ax1.grid(True, alpha=0.3)
        
        # MACD
        ax2.plot(datos.index, macd_data['macd'], label='MACD', linewidth=1.5, color='blue')
        ax2.plot(datos.index, macd_data['signal'], label='Señal', linewidth=1.5, color='red')
        
        # Histograma
        colores = ['green' if x > 0 else 'red' for x in macd_data['histogram']]
        ax2.bar(datos.index, macd_data['histogram'], label='Histograma', color=colores, alpha=0.3)
        
        ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        ax2.set_title('MACD', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Fecha', fontsize=11)
        ax2.set_ylabel('MACD', fontsize=11)
        ax2.legend(loc='best')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    def grafica_rsi(self, datos: pd.DataFrame, nombre_activo: str = None) -> None:
        """
        Grafica el RSI.
        
        Args:
            datos: DataFrame con datos históricos
            nombre_activo: Nombre del activo
        """
        rsi = self.calcular_rsi(datos)
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
        
        # Precio
        ax1.plot(datos.index, datos['Close'], label='Precio', linewidth=2, color='black')
        ax1.set_title(f'{nombre_activo or "Activo"} - Precio', fontsize=12, fontweight='bold')
        ax1.set_ylabel('Precio ($)', fontsize=11)
        ax1.legend(loc='best')
        ax1.grid(True, alpha=0.3)
        
        # RSI
        ax2.plot(datos.index, rsi, label='RSI', linewidth=1.5, color='purple')
        ax2.axhline(y=70, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Sobrecompra (70)')
        ax2.axhline(y=30, color='green', linestyle='--', linewidth=1, alpha=0.7, label='Sobreventa (30)')
        ax2.fill_between(datos.index, 30, 70, alpha=0.1, color='blue')
        
        ax2.set_ylim(0, 100)
        ax2.set_title('RSI (14)', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Fecha', fontsize=11)
        ax2.set_ylabel('RSI', fontsize=11)
        ax2.legend(loc='best')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    def grafica_atr(self, datos: pd.DataFrame, nombre_activo: str = None) -> None:
        """
        Grafica el ATR.
        
        Args:
            datos: DataFrame con datos históricos
            nombre_activo: Nombre del activo
        """
        atr = self.calcular_atr(datos)
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
        
        # Precio
        ax1.plot(datos.index, datos['Close'], label='Precio', linewidth=2, color='black')
        ax1.set_title(f'{nombre_activo or "Activo"} - Precio', fontsize=12, fontweight='bold')
        ax1.set_ylabel('Precio ($)', fontsize=11)
        ax1.legend(loc='best')
        ax1.grid(True, alpha=0.3)
        
        # ATR
        ax2.plot(datos.index, atr, label='ATR', linewidth=1.5, color='orange')
        ax2.fill_between(datos.index, atr, alpha=0.3, color='orange')
        
        ax2.set_title('ATR (14) - Volatilidad', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Fecha', fontsize=11)
        ax2.set_ylabel('ATR', fontsize=11)
        ax2.legend(loc='best')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    def grafica_completa(self, datos: pd.DataFrame, nombre_activo: str = None) -> None:
        """
        Grafica técnica completa con múltiples indicadores.
        
        Args:
            datos: DataFrame con datos históricos
            nombre_activo: Nombre del activo
        """
        # Preparar datos
        sma_20 = self.calcular_sma(datos, 20)
        sma_50 = self.calcular_sma(datos, 50)
        bb = self.calcular_bandas_bollinger(datos)
        rsi = self.calcular_rsi(datos)
        macd_data = self.calcular_macd(datos)
        atr = self.calcular_atr(datos)
        
        fig = plt.figure(figsize=(16, 12))
        gs = fig.add_gridspec(4, 2, hspace=0.3, wspace=0.3)
        
        # Precio y Bandas Bollinger
        ax1 = fig.add_subplot(gs[0, :])
        ax1.fill_between(datos.index, bb['banda_superior'], bb['banda_inferior'], alpha=0.2, color='blue')
        ax1.plot(datos.index, bb['banda_superior'], color='blue', linewidth=0.5, alpha=0.5)
        ax1.plot(datos.index, bb['banda_inferior'], color='blue', linewidth=0.5, alpha=0.5)
        ax1.plot(datos.index, datos['Close'], label='Precio', linewidth=2, color='black')
        ax1.plot(datos.index, sma_20, label='SMA 20', linewidth=1, color='blue', alpha=0.7)
        ax1.plot(datos.index, sma_50, label='SMA 50', linewidth=1, color='red', alpha=0.7)
        ax1.set_title(f'{nombre_activo or "Activo"} - Análisis Técnico Completo', fontsize=14, fontweight='bold')
        ax1.set_ylabel('Precio ($)', fontsize=10)
        ax1.legend(loc='best', fontsize=9)
        ax1.grid(True, alpha=0.3)
        
        # RSI
        ax2 = fig.add_subplot(gs[1, 0])
        ax2.plot(datos.index, rsi, label='RSI', linewidth=1.5, color='purple')
        ax2.axhline(y=70, color='red', linestyle='--', linewidth=1, alpha=0.7)
        ax2.axhline(y=30, color='green', linestyle='--', linewidth=1, alpha=0.7)
        ax2.fill_between(datos.index, 30, 70, alpha=0.1, color='blue')
        ax2.set_ylim(0, 100)
        ax2.set_title('RSI (14)', fontsize=11, fontweight='bold')
        ax2.set_ylabel('RSI', fontsize=10)
        ax2.legend(fontsize=9)
        ax2.grid(True, alpha=0.3)
        
        # MACD
        ax3 = fig.add_subplot(gs[1, 1])
        ax3.plot(datos.index, macd_data['macd'], label='MACD', linewidth=1, color='blue')
        ax3.plot(datos.index, macd_data['signal'], label='Señal', linewidth=1, color='red')
        colores = ['green' if x > 0 else 'red' for x in macd_data['histogram']]
        ax3.bar(datos.index, macd_data['histogram'], label='Histograma', color=colores, alpha=0.3)
        ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        ax3.set_title('MACD', fontsize=11, fontweight='bold')
        ax3.set_ylabel('MACD', fontsize=10)
        ax3.legend(fontsize=9)
        ax3.grid(True, alpha=0.3)
        
        # ATR
        ax4 = fig.add_subplot(gs[2, 0])
        ax4.plot(datos.index, atr, label='ATR', linewidth=1.5, color='orange')
        ax4.fill_between(datos.index, atr, alpha=0.3, color='orange')
        ax4.set_title('ATR (14)', fontsize=11, fontweight='bold')
        ax4.set_ylabel('ATR', fontsize=10)
        ax4.legend(fontsize=9)
        ax4.grid(True, alpha=0.3)
        
        # Volumen (si existe)
        if 'Volume' in datos.columns:
            ax5 = fig.add_subplot(gs[2, 1])
            colores = ['green' if datos['Close'].iloc[i] >= datos['Close'].iloc[i-1] else 'red' 
                      for i in range(1, len(datos))]
            ax5.bar(datos.index[1:], datos['Volume'].iloc[1:], color=colores, alpha=0.6)
            ax5.set_title('Volumen', fontsize=11, fontweight='bold')
            ax5.set_ylabel('Volumen', fontsize=10)
            ax5.grid(True, alpha=0.3)
        
        # Retornos diarios
        ax6 = fig.add_subplot(gs[3, :])
        retornos = datos['Close'].pct_change() * 100
        colores = ['green' if x > 0 else 'red' for x in retornos]
        ax6.bar(datos.index, retornos, color=colores, alpha=0.6)
        ax6.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        ax6.set_title('Retornos Diarios (%)', fontsize=11, fontweight='bold')
        ax6.set_xlabel('Fecha', fontsize=10)
        ax6.set_ylabel('Retorno (%)', fontsize=10)
        ax6.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    def exportar_resumen_tecnico(self, diagnostico: Dict) -> str:
        """
        Genera un resumen formateado del análisis técnico.
        
        Args:
            diagnostico: Diccionario de diagnóstico
        
        Returns:
            String con el resumen
        """
        resumen = f"""
╔════════════════════════════════════════════════════════════╗
║         ANÁLISIS TÉCNICO - {diagnostico.get('nombre', 'ACTIVO'):20}      ║
╚════════════════════════════════════════════════════════════╝

Fecha de análisis: {diagnostico.get('fecha_analisis')}
Tendencia general: {diagnostico.get('tendencia')}
Puntuación: {diagnostico.get('puntuacion')}/10

SEÑALES TÉCNICAS:
"""
        
        for señal in diagnostico.get('señales', []):
            resumen += f"  {señal}\n"
        
        resumen += "\n" + "="*60 + "\n"
        
        return resumen
    
    def __repr__(self) -> str:
        return "AnalizadorTecnico()"

## Interpretación de Indicadores:
| Indicador   | Señal Alcista                          | Señal Bajista                           | Neutral        |
|-------------|-----------------------------------------|-------------------------------------------|----------------|
| **SMA**     | Precio > SMA 20 > SMA 50                | Precio < SMA 20 < SMA 50                  | Cruzamientos   |
| **RSI**     | 30–70 (no extremo)                      | > 70 o < 30                               | 40–60          |
| **MACD**    | Histograma > 0, cruce alcista           | Histograma < 0, cruce bajista             | Plano          |
| **Bollinger** | Precio en banda media–inferior        | Precio en banda media–superior            | Centro         |
| **ATR**     | Bajo a medio                            | Alto                                      | Moderado       |


## Clase 6 — AnalizadorCartera
Frontera eficiente, VaR, contribución al riesgo, correlaciones. Unas 250 líneas. 


In [7]:
import pandas as pd
import numpy as np
from typing import Dict, List, Optional, Tuple
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

class AnalizadorCartera:
    """
    Analiza carteras: frontera eficiente, VaR, correlaciones, contribución al riesgo.
    Optimización de carteras y análisis de riesgo.
    """
    
    def __init__(self, tasa_libre_riesgo: float = 0.02):
        """
        Inicializa el analizador de cartera.
        
        Args:
            tasa_libre_riesgo: Tasa libre de riesgo anual (default: 2%)
        """
        self.tasa_libre_riesgo = tasa_libre_riesgo
        self.cartera_actual = None
        self.retornos_cache = {}
        self.correlacion_cache = None
        sns.set_style("darkgrid")
    
    # ============================================
    # CÁLCULOS FUNDAMENTALES
    # ============================================
    
    def calcular_retornos_diarios(self, datos_dict: Dict[str, pd.DataFrame],
                                 columna: str = 'Close') -> pd.DataFrame:
        """
        Calcula retornos diarios para múltiples activos.
        
        Args:
            datos_dict: Diccionario {ticker: DataFrame}
            columna: Columna de precios a usar
        
        Returns:
            DataFrame con retornos diarios
        """
        retornos_dict = {}
        
        for ticker, datos in datos_dict.items():
            if columna not in datos.columns:
                print(f"⚠️  {ticker}: Columna '{columna}' no encontrada")
                continue
            
            precios = datos[columna]
            retornos = np.log(precios / precios.shift(1)).dropna()
            retornos_dict[ticker] = retornos
        
        # Alinear índices y crear DataFrame
        retornos_df = pd.DataFrame(retornos_dict)
        retornos_df = retornos_df.dropna()
        
        print(f"✓ Retornos calculados: {len(retornos_df)} días")
        return retornos_df
    
    def calcular_matriz_correlacion(self, retornos: pd.DataFrame) -> pd.DataFrame:
        """
        Calcula la matriz de correlación.
        
        Args:
            retornos: DataFrame de retornos diarios
        
        Returns:
            DataFrame con matriz de correlación
        """
        correlacion = retornos.corr()
        self.correlacion_cache = correlacion
        return correlacion
    
    def calcular_retornos_esperados(self, retornos: pd.DataFrame,
                                   dias_trading: int = 252) -> pd.Series:
        """
        Calcula retornos esperados anualizados.
        
        Args:
            retornos: DataFrame de retornos diarios
            dias_trading: Días de trading por año
        
        Returns:
            Series con retornos anualizados
        """
        retornos_diarios_medios = retornos.mean()
        retornos_anualizados = retornos_diarios_medios * dias_trading
        
        return retornos_anualizados
    
    def calcular_matriz_covarianza(self, retornos: pd.DataFrame,
                                  dias_trading: int = 252) -> pd.DataFrame:
        """
        Calcula la matriz de covarianza anualizada.
        
        Args:
            retornos: DataFrame de retornos diarios
            dias_trading: Días de trading por año
        
        Returns:
            DataFrame con matriz de covarianza anualizada
        """
        covarianza_diaria = retornos.cov()
        covarianza_anualizada = covarianza_diaria * dias_trading
        
        return covarianza_anualizada
    
    def calcular_volatilidades(self, retornos: pd.DataFrame,
                              dias_trading: int = 252) -> pd.Series:
        """
        Calcula volatilidades anualizadas para cada activo.
        
        Args:
            retornos: DataFrame de retornos diarios
            dias_trading: Días de trading por año
        
        Returns:
            Series con volatilidades
        """
        volatilidades_diarias = retornos.std()
        volatilidades = volatilidades_diarias * np.sqrt(dias_trading)
        
        return volatilidades
    
    # ============================================
    # CARTERA ACTUAL
    # ============================================
    
    def establecer_cartera(self, pesos: Dict[str, float], 
                          retornos: pd.DataFrame) -> Dict:
        """
        Establece una cartera con pesos específicos.
        
        Args:
            pesos: Diccionario {ticker: peso}
            retornos: DataFrame de retornos diarios
        
        Returns:
            Diccionario con características de la cartera
        """
        # Validar pesos
        peso_total = sum(pesos.values())
        if not (0.99 <= peso_total <= 1.01):
            raise ValueError(f"Pesos no suman 1.0, suma: {peso_total}")
        
        # Crear vector de pesos
        tickers = list(pesos.keys())
        w = np.array([pesos[t] for t in tickers])
        
        # Calcular características
        retornos_esperados = self.calcular_retornos_esperados(retornos[tickers])
        covarianza = self.calcular_matriz_covarianza(retornos[tickers])
        
        retorno_cartera = np.sum(w * retornos_esperados)
        volatilidad_cartera = np.sqrt(np.dot(w, np.dot(covarianza, w)))
        
        self.cartera_actual = {
            'pesos': pesos,
            'tickers': tickers,
            'w': w,
            'retorno': retorno_cartera,
            'volatilidad': volatilidad_cartera,
            'sharpe': (retorno_cartera - self.tasa_libre_riesgo) / volatilidad_cartera,
            'retornos_esperados': retornos_esperados,
            'covarianza': covarianza
        }
        
        return self.cartera_actual
    
    # ============================================
    # OPTIMIZACIÓN
    # ============================================
    
    def frontera_eficiente(self, retornos: pd.DataFrame,
                          num_carteras: int = 100) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Calcula la frontera eficiente generando carteras aleatorias.
        
        Args:
            retornos: DataFrame de retornos diarios
            num_carteras: Número de carteras a simular
        
        Returns:
            Tupla (volatilidades, retornos, sharpes)
        """
        tickers = retornos.columns.tolist()
        n_activos = len(tickers)
        
        retornos_esperados = self.calcular_retornos_esperados(retornos)
        covarianza = self.calcular_matriz_covarianza(retornos)
        
        resultados = np.zeros((num_carteras, 3))
        pesos_aleatorios = []
        
        print(f"📊 Generando {num_carteras} carteras aleatorias...")
        
        for i in range(num_cartelas):
            # Pesos aleatorios
            w = np.random.random(n_activos)
            w = w / w.sum()
            pesos_aleatorios.append(w)
            
            # Cálculos
            retorno = np.sum(w * retornos_esperados)
            volatilidad = np.sqrt(np.dot(w, np.dot(covarianza, w)))
            sharpe = (retorno - self.tasa_libre_riesgo) / volatilidad if volatilidad > 0 else 0
            
            resultados[i] = [volatilidad, retorno, sharpe]
        
        return resultados[:, 0], resultados[:, 1], resultados[:, 2]
    
    def cartera_minima_varianza(self, retornos: pd.DataFrame) -> Dict:
        """
        Calcula la cartera de mínima varianza.
        
        Args:
            retornos: DataFrame de retornos diarios
        
        Returns:
            Diccionario con características de la cartera
        """
        tickers = retornos.columns.tolist()
        n_activos = len(tickers)
        
        covarianza = self.calcular_matriz_covarianza(retornos)
        retornos_esperados = self.calcular_retornos_esperados(retornos)
        
        # Función objetivo: minimizar varianza
        def varianza_cartera(w):
            return np.dot(w, np.dot(covarianza, w))
        
        # Restricciones: pesos suman 1
        restricciones = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
        
        # Límites: 0 <= w <= 1
        limites = tuple((0, 1) for _ in range(n_activos))
        
        # Optimizar
        resultado = minimize(
            varianza_cartera,
            x0=np.array([1/n_activos] * n_activos),
            method='SLSQP',
            bounds=limites,
            constraints=restricciones
        )
        
        w_optimo = resultado.x
        pesos = {tickers[i]: w_optimo[i] for i in range(n_activos)}
        
        return self.establecer_cartera(pesos, retornos)
    
    def cartera_maximo_sharpe(self, retornos: pd.DataFrame) -> Dict:
        """
        Calcula la cartera de máximo Sharpe Ratio.
        
        Args:
            retornos: DataFrame de retornos diarios
        
        Returns:
            Diccionario con características de la cartera
        """
        tickers = retornos.columns.tolist()
        n_activos = len(tickers)
        
        covarianza = self.calcular_matriz_covarianza(retornos)
        retornos_esperados = self.calcular_retornos_esperados(retornos)
        
        # Función objetivo: maximizar Sharpe (minimizar negativo)
        def neg_sharpe(w):
            retorno = np.sum(w * retornos_esperados)
            volatilidad = np.sqrt(np.dot(w, np.dot(covarianza, w)))
            sharpe = (retorno - self.tasa_libre_riesgo) / volatilidad if volatilidad > 0 else 0
            return -sharpe
        
        # Restricciones: pesos suman 1
        restricciones = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
        
        # Límites: 0 <= w <= 1
        limites = tuple((0, 1) for _ in range(n_activos))
        
        # Optimizar
        resultado = minimize(
            neg_sharpe,
            x0=np.array([1/n_activos] * n_activos),
            method='SLSQP',
            bounds=limites,
            constraints=restricciones
        )
        
        w_optimo = resultado.x
        pesos = {tickers[i]: w_optimo[i] for i in range(n_activos)}
        
        return self.establecer_cartera(pesos, retornos)
    
    def cartera_maxima_rentabilidad(self, retornos: pd.DataFrame) -> Dict:
        """
        Calcula la cartera de máxima rentabilidad esperada.
        
        Args:
            retornos: DataFrame de retornos diarios
        
        Returns:
            Diccionario con características de la cartera
        """
        tickers = retornos.columns.tolist()
        n_activos = len(tickers)
        
        retornos_esperados = self.calcular_retornos_esperados(retornos)
        
        # Función objetivo: maximizar retorno (minimizar negativo)
        def neg_retorno(w):
            return -np.sum(w * retornos_esperados)
        
        # Restricciones: pesos suman 1
        restricciones = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
        
        # Límites: 0 <= w <= 1
        limites = tuple((0, 1) for _ in range(n_activos))
        
        # Optimizar
        resultado = minimize(
            neg_retorno,
            x0=np.array([1/n_activos] * n_activos),
            method='SLSQP',
            bounds=limites,
            constraints=restricciones
        )
        
        w_optimo = resultado.x
        pesos = {tickers[i]: w_optimo[i] for i in range(n_activos)}
        
        return self.establecer_cartera(pesos, retornos)
    
    # ============================================
    # RIESGO
    # ============================================
    
    def calcular_var_cartera(self, retornos_cartera: pd.Series,
                            nivel_confianza: float = 0.95) -> float:
        """
        Calcula el Value at Risk de la cartera.
        
        Args:
            retornos_cartera: Series de retornos de la cartera
            nivel_confianza: Nivel de confianza
        
        Returns:
            VaR en decimal
        """
        var = np.percentile(retornos_cartera, (1 - nivel_confianza) * 100)
        return var
    
    def calcular_cvar_cartera(self, retornos_cartera: pd.Series,
                             nivel_confianza: float = 0.95) -> float:
        """
        Calcula el Conditional VaR (Expected Shortfall) de la cartera.
        
        Args:
            retornos_cartera: Series de retornos de la cartera
            nivel_confianza: Nivel de confianza
        
        Returns:
            CVaR en decimal
        """
        var = self.calcular_var_cartera(retornos_cartera, nivel_confianza)
        cvar = retornos_cartera[retornos_cartera <= var].mean()
        return cvar
    
    def calcular_contribucion_riesgo(self, retornos: pd.DataFrame) -> pd.Series:
        """
        Calcula la contribución marginal de cada activo al riesgo de la cartera.
        
        Requiere una cartera establecida con establecer_cartera().
        
        Args:
            retornos: DataFrame de retornos diarios
        
        Returns:
            Series con contribución de riesgo por activo
        """
        if self.cartera_actual is None:
            raise ValueError("Debe establecer una cartera primero")
        
        w = self.cartera_actual['w']
        covarianza = self.cartera_actual['covarianza']
        volatilidad = self.cartera_actual['volatilidad']
        
        # Contribución marginal al riesgo
        mcr = np.dot(covarianza, w) / volatilidad
        
        # Contribución de riesgo
        contribucion = w * mcr
        
        return pd.Series(contribucion, index=self.cartera_actual['tickers'])
    
    def calcular_diversificacion(self, retornos: pd.DataFrame) -> float:
        """
        Calcula el ratio de diversificación de la cartera.
        
        Requiere una cartera establecida con establecer_cartera().
        
        Args:
            retornos: DataFrame de retornos diarios
        
        Returns:
            Ratio de diversificación (0-1, mayor es mejor)
        """
        if self.cartera_actual is None:
            raise ValueError("Debe establecer una cartera primero")
        
        w = self.cartera_actual['w']
        volatilidades = self.calcular_volatilidades(retornos[self.cartera_actual['tickers']])
        volatilidad_cartera = self.cartera_actual['volatilidad']
        
        # Volatilidad ponderada
        vol_ponderada = np.sum(w * volatilidades)
        
        # Ratio de diversificación
        ratio_diversificacion = vol_ponderada / volatilidad_cartera
        
        return ratio_diversificacion
    
    # ============================================
    # GRÁFICAS
    # ============================================
    
    def grafica_frontera_eficiente(self, retornos: pd.DataFrame,
                                  num_carteras: int = 1000) -> None:
        """
        Grafica la frontera eficiente.
        
        Args:
            retornos: DataFrame de retornos diarios
            num_carteras: Número de carteras aleatorias a simular
        """
        vols, rets, sharpes = self.frontera_eficiente(retornos, num_carteras)
        
        plt.figure(figsize=(12, 8))
        
        # Scatter plot coloreado por Sharpe
        scatter = plt.scatter(vols, rets, c=sharpes, cmap='viridis', 
                             alpha=0.6, s=30, edgecolors='black', linewidth=0.5)
        
        # Cartera actual si existe
        if self.cartera_actual:
            plt.scatter(self.cartera_actual['volatilidad'], 
                       self.cartera_actual['retorno'],
                       marker='*', s=1000, c='red', 
                       edgecolors='black', linewidth=2,
                       label='Cartera Actual', zorder=5)
        
        # Etiquetas
        plt.xlabel('Volatilidad (Riesgo)', fontsize=12, fontweight='bold')
        plt.ylabel('Retorno Esperado', fontsize=12, fontweight='bold')
        plt.title('Frontera Eficiente de Carteras', fontsize=14, fontweight='bold')
        plt.colorbar(scatter, label='Sharpe Ratio')
        plt.legend(loc='best')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    def grafica_correlacion(self, retornos: pd.DataFrame) -> None:
        """
        Grafica la matriz de correlación como heatmap.
        
        Args:
            retornos: DataFrame de retornos diarios
        """
        correlacion = self.calcular_matriz_correlacion(retornos)
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(correlacion, annot=True, fmt='.2f', cmap='coolwarm',
                   center=0, vmin=-1, vmax=1, square=True,
                   cbar_kws={'label': 'Correlación'})
        plt.title('Matriz de Correlación de Activos', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
    
    def grafica_pesos_cartera(self, titulo: str = "Pesos de la Cartera") -> None:
        """
        Grafica los pesos de la cartera actual.
        
        Args:
            titulo: Título de la gráfica
        """
        if self.cartera_actual is None:
            print("❌ No hay cartera establecida")
            return
        
        pesos = self.cartera_actual['pesos']
        tickers = list(pesos.keys())
        valores = list(pesos.values())
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Gráfica de pastel
        colores = plt.cm.Set3(np.linspace(0, 1, len(tickers)))
        ax1.pie(valores, labels=tickers, autopct='%1.1f%%', colors=colores,
               startangle=90, textprops={'fontsize': 11})
        ax1.set_title(titulo, fontsize=12, fontweight='bold')
        
        # Gráfica de barras
        ax2.bar(tickers, valores, color=colores, edgecolor='black', linewidth=1.5)
        ax2.set_ylabel('Peso', fontsize=11)
        ax2.set_title(f'{titulo} (Barras)', fontsize=12, fontweight='bold')
        ax2.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.show()
    
    def grafica_contribucion_riesgo(self, retornos: pd.DataFrame) -> None:
        """
        Grafica la contribución de cada activo al riesgo.
        
        Args:
            retornos: DataFrame de retornos diarios
        """
        if self.cartera_actual is None:
            print("❌ No hay cartera establecida")
            return
        
        contribucion = self.calcular_contribucion_riesgo(retornos)
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Gráfica de barras
        colores = ['red' if x > 0 else 'blue' for x in contribucion]
        ax1.barh(contribucion.index, contribucion.values, color=colores, edgecolor='black')
        ax1.set_xlabel('Contribución al Riesgo', fontsize=11)
        ax1.set_title('Contribución Marginal de Riesgo', fontsize=12, fontweight='bold')
        ax1.grid(True, alpha=0.3, axis='x')
        
        # Gráfica de pastel
        contribucion_abs = abs(contribucion)
        ax2.pie(contribucion_abs, labels=contribucion.index, autopct='%1.1f%%',
               startangle=90, textprops={'fontsize': 11})
        ax2.set_title('Riesgo Relativo por Activo', fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        plt.show()
    
    def grafica_comparacion_carteras(self, carteras: Dict, retornos: pd.DataFrame) -> None:
        """
        Compara múltiples carteras en el plano riesgo-retorno.
        
        Args:
            carteras: Diccionario {nombre: pesos_dict}
            retornos: DataFrame de retornos diarios
        """
        volatilidades = []
        retornos_esperados = []
        sharpes = []
        nombres = []
        
        for nombre, pesos in carteras.items():
            cartera = self.establecer_cartera(pesos, retornos)
            volatilidades.append(cartera['volatilidad'])
            retornos_esperados.append(cartera['retorno'])
            sharpes.append(cartera['sharpe'])
            nombres.append(nombre)
        
        plt.figure(figsize=(12, 8))
        
        colores = plt.cm.Set1(np.linspace(0, 1, len(nombres)))
        scatter = plt.scatter(volatilidades, retornos_esperados, s=200, 
                             c=colores, edgecolors='black', linewidth=2, zorder=5)
        
        for i, nombre in enumerate(nombres):
            plt.annotate(nombre, (volatilidades[i], retornos_esperados[i]),
                        xytext=(10, 5), textcoords='offset points', fontsize=10)
        
        plt.xlabel('Volatilidad (Riesgo)', fontsize=12, fontweight='bold')
        plt.ylabel('Retorno Esperado', fontsize=12, fontweight='bold')
        plt.title('Comparación de Carteras', fontsize=14, fontweight='bold')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    # ============================================
    # REPORTES
    # ============================================
    
    def exportar_resumen_cartera(self) -> str:
        """
        Genera un resumen formateado de la cartera actual.
        
        Returns:
            String con el resumen
        """
        if self.cartera_actual is None:
            return "❌ No hay cartera establecida"
        
        cartera = self.cartera_actual
        
        resumen = f"""
╔════════════════════════════════════════════════════════════╗
║              ANÁLISIS DE CARTERA                           ║
╚════════════════════════════════════════════════════════════╝

COMPOSICIÓN DE LA CARTERA:
"""
        
        resumen += "┌──────────┬──────────┐\n"
        resumen += "│ Ticker   │ Peso     │\n"
        resumen += "├──────────┼──────────┤\n"
        
        for ticker, peso in cartera['pesos'].items():
            resumen += f"│ {ticker:8} │ {peso:7.2%} │\n"
        
        resumen += "└──────────┴──────────┘\n"
        
        resumen += f"""
CARACTERÍSTICAS DE LA CARTERA:
  Retorno esperado: {cartera['retorno']:.2%}
  Volatilidad: {cartera['volatilidad']:.2%}
  Sharpe Ratio: {cartera['sharpe']:.4f}

MATRIZ DE CORRELACIONES:
"""
        
        correlacion = cartera['covarianza'] / (
            np.outer(
                np.sqrt(np.diag(cartera['covarianza'])),
                np.sqrt(np.diag(cartera['covarianza']))
            )
        )
        
        resumen += str(pd.DataFrame(correlacion, 
                                   index=cartera['tickers'],
                                   columns=cartera['tickers']).round(3))
        
        return resumen
    
    def exportar_comparacion_carteras_optimas(self, retornos: pd.DataFrame) -> str:
        """
        Compara las tres carteras óptimas principales.
        
        Args:
            retornos: DataFrame de retornos diarios
        
        Returns:
            String con comparación
        """
        print("Calculando carteras óptimas...")
        
        cartera_min_var = self.cartera_minima_varianza(retornos)
        cartera_max_sharpe = self.cartera_maximo_sharpe(retornos)
        cartera_max_ret = self.cartera_maxima_rentabilidad(retornos)
        
        resumen = f"""
╔════════════════════════════════════════════════════════════╗
║         COMPARACIÓN DE CARTERAS ÓPTIMAS                   ║
╚════════════════════════════════════════════════════════════╝

┌──────────────────┬──────────────────┬──────────────────┐
│ Mínima Varianza  │ Máximo Sharpe    │ Máximo Retorno   │
├──────────────────┼──────────────────┼──────────────────┤
│ Retorno: {cartera_min_var['retorno']:.2%}     │ Retorno: {cartera_max_sharpe['retorno']:.2%}    │ Retorno: {cartera_max_ret['retorno']:.2%}    │
│ Riesgo: {cartera_min_var['volatilidad']:.2%}      │ Riesgo: {cartera_max_sharpe['volatilidad']:.2%}     │ Riesgo: {cartera_max_ret['volatilidad']:.2%}     │
│ Sharpe: {cartera_min_var['sharpe']:.4f}      │ Sharpe: {cartera_max_sharpe['sharpe']:.4f}     │ Sharpe: {cartera_max_ret['sharpe']:.4f}     │
└──────────────────┴──────────────────┴──────────────────┘

RECOMENDACIÓN:
  → Cartera de Máximo Sharpe: Mejor relación riesgo-retorno
  → Cartera de Mínima Varianza: Mínimo riesgo
  → Cartera de Máximo Retorno: Máximo rendimiento (mayor riesgo)
"""
        
        return resumen
    
    def __repr__(self) -> str:
        if self.cartera_actual:
            return f"AnalizadorCartera(retorno={self.cartera_actual['retorno']:.2%}, riesgo={self.cartera_actual['volatilidad']:.2%})"
        return "AnalizadorCartera()"

## Métricas Clave de Cartera:

| Métrica              | Interpretación                                                |
|----------------------|----------------------------------------------------------------|
| **Sharpe Ratio**     | > 1.0: Excelente relación riesgo‑retorno                       |
| **Correlación**      | –1 a 1; cercano a 0: mejor diversificación                     |
| **VaR (95%)**        | Pérdida máxima esperada en el 5% de los peores días            |
| **CVaR**             | Pérdida promedio cuando se supera el VaR                       |
| **Diversificación**  | > 1.2: bien diversificada                                      |
| **Contribución Riesgo** | Qué porcentaje del riesgo aporta cada activo               |


## Clase 7 — Exportador
PDF, Excel, heatmaps, gráfico subacuático. Unas 300 líneas. 


In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
import warnings
warnings.filterwarnings('ignore')

class Exportador:
    """
    Exporta análisis a PDF, Excel y genera gráficas profesionales.
    Incluye heatmaps, gráficos subacuáticos y reportes completos.
    """
    
    def __init__(self, gestor_rutas):
        """
        Inicializa el exportador.
        
        Args:
            gestor_rutas: Instancia de GestorRutas
        """
        self.gestor_rutas = gestor_rutas
        self.ruta_exportaciones = gestor_rutas.obtener_ruta("exportaciones")
        self.ruta_exportaciones.mkdir(parents=True, exist_ok=True)
        sns.set_style("darkgrid")
    
    # ============================================
    # EXPORTAR A EXCEL
    # ============================================
    
    def exportar_metricas_excel(self, metricas_dict: Dict[str, Dict],
                               nombre_archivo: str = "Metricas_Activos.xlsx") -> Path:
        """
        Exporta métricas de múltiples activos a Excel.
        
        Args:
            metricas_dict: Diccionario {ticker: metricas_dict}
            nombre_archivo: Nombre del archivo
        
        Returns:
            Path del archivo creado
        """
        # Convertir a DataFrame
        df = pd.DataFrame(metricas_dict).T
        
        # Crear archivo Excel con estilos
        ruta_archivo = self.ruta_exportaciones / nombre_archivo
        
        with pd.ExcelWriter(ruta_archivo, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name='Métricas')
            
            # Aplicar estilos
            worksheet = writer.sheets['Métricas']
            self._formatear_worksheet_excel(worksheet)
        
        print(f"✓ Métricas exportadas: {ruta_archivo}")
        return ruta_archivo
    
    def exportar_cartera_excel(self, cartera_df: pd.DataFrame,
                              metricas: Dict = None,
                              nombre_archivo: str = "Cartera_Analisis.xlsx") -> Path:
        """
        Exporta análisis completo de cartera a Excel.
        
        Args:
            cartera_df: DataFrame con composición de cartera
            metricas: Diccionario con métricas de cartera
            nombre_archivo: Nombre del archivo
        
        Returns:
            Path del archivo creado
        """
        ruta_archivo = self.ruta_exportaciones / nombre_archivo
        
        with pd.ExcelWriter(ruta_archivo, engine='openpyxl') as writer:
            # Hoja 1: Composición
            cartera_df.to_excel(writer, sheet_name='Composición', index=True)
            
            # Hoja 2: Métricas
            if metricas:
                metricas_df = pd.DataFrame(
                    list(metricas.items()),
                    columns=['Métrica', 'Valor']
                )
                metricas_df.to_excel(writer, sheet_name='Métricas', index=False)
            
            # Formatear
            for sheet_name in writer.sheets:
                worksheet = writer.sheets[sheet_name]
                self._formatear_worksheet_excel(worksheet)
        
        print(f"✓ Cartera exportada: {ruta_archivo}")
        return ruta_archivo
    
    def exportar_correlaciones_excel(self, correlacion_df: pd.DataFrame,
                                    nombre_archivo: str = "Correlaciones.xlsx") -> Path:
        """
        Exporta matriz de correlación a Excel.
        
        Args:
            correlacion_df: DataFrame con matriz de correlación
            nombre_archivo: Nombre del archivo
        
        Returns:
            Path del archivo creado
        """
        ruta_archivo = self.ruta_exportaciones / nombre_archivo
        
        with pd.ExcelWriter(ruta_archivo, engine='openpyxl') as writer:
            correlacion_df.to_excel(writer, sheet_name='Correlación')
            
            worksheet = writer.sheets['Correlación']
            self._formatear_worksheet_excel(worksheet)
        
        print(f"✓ Correlaciones exportadas: {ruta_archivo}")
        return ruta_archivo
    
    def exportar_comparativa_excel(self, comparativa_df: pd.DataFrame,
                                  nombre_archivo: str = "Comparativa_Carteras.xlsx") -> Path:
        """
        Exporta comparativa de carteras a Excel.
        
        Args:
            comparativa_df: DataFrame con comparativa
            nombre_archivo: Nombre del archivo
        
        Returns:
            Path del archivo creado
        """
        ruta_archivo = self.ruta_exportaciones / nombre_archivo
        
        with pd.ExcelWriter(ruta_archivo, engine='openpyxl') as writer:
            comparativa_df.to_excel(writer, sheet_name='Comparativa', index=False)
            
            worksheet = writer.sheets['Comparativa']
            self._formatear_worksheet_excel(worksheet)
        
        print(f"✓ Comparativa exportada: {ruta_archivo}")
        return ruta_archivo
    
    # ============================================
    # GRÁFICAS AVANZADAS
    # ============================================
    
    def grafica_heatmap_correlacion(self, correlacion_df: pd.DataFrame,
                                   guardar: bool = True) -> Optional[Path]:
        """
        Crea un heatmap profesional de correlación.
        
        Args:
            correlacion_df: DataFrame con matriz de correlación
            guardar: Si True, guarda la imagen
        
        Returns:
            Path de la imagen (si guardar=True)
        """
        plt.figure(figsize=(12, 10))
        
        # Crear heatmap
        sns.heatmap(correlacion_df, annot=True, fmt='.2f',
                   cmap='RdYlGn', center=0, vmin=-1, vmax=1,
                   square=True, linewidths=1, cbar_kws={'label': 'Correlación'},
                   annot_kws={'size': 9})
        
        plt.title('Matriz de Correlación de Activos', fontsize=14, fontweight='bold', pad=20)
        plt.tight_layout()
        
        if guardar:
            ruta_imagen = self.ruta_exportaciones / "heatmap_correlacion.png"
            plt.savefig(ruta_imagen, dpi=300, bbox_inches='tight')
            print(f"✓ Heatmap guardado: {ruta_imagen}")
            plt.close()
            return ruta_imagen
        else:
            plt.show()
            return None
    
    def grafica_grafico_subacuatico(self, datos: pd.DataFrame, columna: str = 'Close',
                                   guardar: bool = True) -> Optional[Path]:
        """
        Crea un gráfico de barras "subacuático" mostrando períodos positivos/negativos.
        
        Args:
            datos: DataFrame con datos históricos
            columna: Columna de precios
            guardar: Si True, guarda la imagen
        
        Returns:
            Path de la imagen (si guardar=True)
        """
        retornos = datos[columna].pct_change() * 100
        retorno_acumulado = (1 + retornos / 100).cumprod() - 1
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
        
        # Gráfica superior: Retorno acumulado
        ax1.fill_between(datos.index, retorno_acumulado, 0, 
                        where=(retorno_acumulado >= 0), alpha=0.6,
                        color='green', label='Positivo', interpolate=True)
        ax1.fill_between(datos.index, retorno_acumulado, 0,
                        where=(retorno_acumulado < 0), alpha=0.6,
                        color='red', label='Negativo', interpolate=True)
        
        ax1.plot(datos.index, retorno_acumulado, color='black', linewidth=2)
        ax1.axhline(y=0, color='black', linestyle='-', linewidth=1)
        ax1.set_title('Retorno Acumulado - Gráfico Subacuático', fontsize=12, fontweight='bold')
        ax1.set_ylabel('Retorno Acumulado', fontsize=11)
        ax1.legend(loc='best')
        ax1.grid(True, alpha=0.3)
        
        # Gráfica inferior: Drawdown
        valor_maximo = datos[columna].expanding().max()
        drawdown = (datos[columna] - valor_maximo) / valor_maximo * 100
        
        ax2.fill_between(datos.index, drawdown, 0, alpha=0.6, color='red')
        ax2.set_title('Drawdown (Pérdida desde Máximo)', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Fecha', fontsize=11)
        ax2.set_ylabel('Drawdown (%)', fontsize=11)
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        if guardar:
            ruta_imagen = self.ruta_exportaciones / "grafico_subacuatico.png"
            plt.savefig(ruta_imagen, dpi=300, bbox_inches='tight')
            print(f"✓ Gráfico subacuático guardado: {ruta_imagen}")
            plt.close()
            return ruta_imagen
        else:
            plt.show()
            return None
    
    def grafica_retornos_distribucion(self, retornos: pd.Series, nombre: str = "Activo",
                                     guardar: bool = True) -> Optional[Path]:
        """
        Crea histograma de distribución de retornos con estadísticas.
        
        Args:
            retornos: Series de retornos diarios
            nombre: Nombre del activo
            guardar: Si True, guarda la imagen
        
        Returns:
            Path de la imagen (si guardar=True)
        """
        fig, ax = plt.subplots(figsize=(12, 6))
        
        # Histograma
        n, bins, patches = ax.hist(retornos * 100, bins=50, edgecolor='black',
                                   alpha=0.7, color='skyblue')
        
        # Colorear según positivo/negativo
        for i, patch in enumerate(patches):
            if bins[i] < 0:
                patch.set_facecolor('red')
            else:
                patch.set_facecolor('green')
        
        # Líneas de referencia
        media = retornos.mean() * 100
        std = retornos.std() * 100
        
        ax.axvline(media, color='black', linestyle='--', linewidth=2, label=f'Media: {media:.3f}%')
        ax.axvline(media + std, color='orange', linestyle='--', linewidth=1, alpha=0.7, label=f'+1σ: {media+std:.3f}%')
        ax.axvline(media - std, color='orange', linestyle='--', linewidth=1, alpha=0.7, label=f'-1σ: {media-std:.3f}%')
        
        ax.set_title(f'Distribución de Retornos - {nombre}', fontsize=14, fontweight='bold')
        ax.set_xlabel('Retorno Diario (%)', fontsize=12)
        ax.set_ylabel('Frecuencia', fontsize=12)
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        
        if guardar:
            ruta_imagen = self.ruta_exportaciones / f"distribucion_retornos_{nombre}.png"
            plt.savefig(ruta_imagen, dpi=300, bbox_inches='tight')
            print(f"✓ Distribución de retornos guardada: {ruta_imagen}")
            plt.close()
            return ruta_imagen
        else:
            plt.show()
            return None
    
    def grafica_composicion_cartera(self, pesos: Dict[str, float],
                                   guardar: bool = True) -> Optional[Path]:
        """
        Crea gráfica de composición de cartera.
        
        Args:
            pesos: Diccionario {ticker: peso}
            guardar: Si True, guarda la imagen
        
        Returns:
            Path de la imagen (si guardar=True)
        """
        tickers = list(pesos.keys())
        valores = list(pesos.values())
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        
        colores = plt.cm.Set3(np.linspace(0, 1, len(tickers)))
        
        # Gráfica de pastel
        wedges, texts, autotexts = ax1.pie(valores, labels=tickers, autopct='%1.1f%%',
                                            colors=colores, startangle=90,
                                            textprops={'fontsize': 10})
        
        for autotext in autotexts:
            autotext.set_color('white')
            autotext.set_fontweight('bold')
        
        ax1.set_title('Composición de Cartera (Pastel)', fontsize=12, fontweight='bold')
        
        # Gráfica de barras
        ax2.barh(tickers, valores, color=colores, edgecolor='black', linewidth=1.5)
        ax2.set_xlabel('Peso', fontsize=11)
        ax2.set_title('Composición de Cartera (Barras)', fontsize=12, fontweight='bold')
        ax2.grid(True, alpha=0.3, axis='x')
        
        # Añadir porcentajes en las barras
        for i, (ticker, valor) in enumerate(zip(tickers, valores)):
            ax2.text(valor + 0.01, i, f'{valor:.1%}', va='center', fontweight='bold')
        
        plt.tight_layout()
        
        if guardar:
            ruta_imagen = self.ruta_exportaciones / "composicion_cartera.png"
            plt.savefig(ruta_imagen, dpi=300, bbox_inches='tight')
            print(f"✓ Composición de cartera guardada: {ruta_imagen}")
            plt.close()
            return ruta_imagen
        else:
            plt.show()
            return None
    
    def grafica_riesgo_retorno(self, carteras: Dict[str, Dict], 
                              guardar: bool = True) -> Optional[Path]:
        """
        Crea gráfica de riesgo vs retorno de múltiples carteras.
        
        Args:
            carteras: Diccionario {nombre: {'riesgo': float, 'retorno': float, ...}}
            guardar: Si True, guarda la imagen
        
        Returns:
            Path de la imagen (si guardar=True)
        """
        nombres = []
        riesgos = []
        retornos = []
        sharpes = []
        
        for nombre, data in carteras.items():
            nombres.append(nombre)
            riesgos.append(data.get('volatilidad', data.get('riesgo', 0)))
            retornos.append(data.get('retorno', 0))
            sharpes.append(data.get('sharpe', 0))
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Scatter plot coloreado por Sharpe
        scatter = ax.scatter(riesgos, retornos, s=500, c=sharpes, cmap='viridis',
                            alpha=0.6, edgecolors='black', linewidth=2, zorder=5)
        
        # Etiquetas
        for i, nombre in enumerate(nombres):
            ax.annotate(nombre, (riesgos[i], retornos[i]),
                       xytext=(10, 10), textcoords='offset points',
                       fontsize=10, fontweight='bold',
                       bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.5))
        
        # Línea de capital (CML) si hay datos suficientes
        if len(riesgos) >= 2:
            z = np.polyfit(riesgos, retornos, 1)
            p = np.poly1d(z)
            x_line = np.linspace(min(riesgos) * 0.9, max(riesgos) * 1.1, 100)
            ax.plot(x_line, p(x_line), 'r--', alpha=0.5, linewidth=2, label='Capital Market Line')
        
        ax.set_xlabel('Riesgo (Volatilidad)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Retorno Esperado', fontsize=12, fontweight='bold')
        ax.set_title('Riesgo vs Retorno de Carteras', fontsize=14, fontweight='bold')
        
        cbar = plt.colorbar(scatter, ax=ax, label='Sharpe Ratio')
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        if guardar:
            ruta_imagen = self.ruta_exportaciones / "riesgo_retorno.png"
            plt.savefig(ruta_imagen, dpi=300, bbox_inches='tight')
            print(f"✓ Gráfica riesgo-retorno guardada: {ruta_imagen}")
            plt.close()
            return ruta_imagen
        else:
            plt.show()
            return None
    
    # ============================================
    # EXPORTAR A PDF
    # ============================================
    
    def exportar_pdf_completo(self, datos_dict: Dict, metricas_dict: Dict,
                             correlacion_df: pd.DataFrame,
                             cartera_pesos: Dict,
                             nombre_archivo: str = "Reporte_Completo.pdf") -> Path:
        """
        Genera un PDF completo con múltiples secciones.
        
        Args:
            datos_dict: Diccionario {ticker: DataFrame}
            metricas_dict: Diccionario {ticker: metricas}
            correlacion_df: DataFrame con matriz de correlación
            cartera_pesos: Diccionario con pesos de la cartera
            nombre_archivo: Nombre del PDF
        
        Returns:
            Path del PDF creado
        """
        ruta_pdf = self.ruta_exportaciones / nombre_archivo
        
        with PdfPages(ruta_pdf) as pdf:
            # Página 1: Portada
            self._pdf_portada(pdf)
            
            # Página 2: Tabla de contenidos
            self._pdf_tabla_contenidos(pdf)
            
            # Páginas 3+: Análisis de tickers
            self._pdf_analisis_tickers(pdf, metricas_dict)
            
            # Páginas: Correlaciones
            self._pdf_seccion_correlaciones(pdf, correlacion_df)
            
            # Páginas: Composición de cartera
            self._pdf_seccion_cartera(pdf, cartera_pesos)
            
            # Metadata
            d = pdf.infodict()
            d['Title'] = 'Reporte de Análisis de Cartera'
            d['Author'] = 'Analizador de Cartera'
            d['Subject'] = 'Análisis técnico y fundamental'
            d['Keywords'] = 'Cartera, Análisis, Finanzas'
            d['CreationDate'] = datetime.now()
        
        print(f"✓ PDF completo generado: {ruta_pdf}")
        return ruta_pdf
    
    def _pdf_portada(self, pdf: PdfPages) -> None:
        """Crea página de portada del PDF."""
        fig = plt.figure(figsize=(8.5, 11))
        ax = fig.add_subplot(111)
        ax.axis('off')
        
        # Título
        ax.text(0.5, 0.8, 'REPORTE DE ANÁLISIS DE CARTERA', 
               ha='center', fontsize=28, fontweight='bold', transform=ax.transAxes)
        
        # Subtítulo
        ax.text(0.5, 0.7, 'Análisis Técnico y Fundamental', 
               ha='center', fontsize=16, style='italic', transform=ax.transAxes)
        
        # Información
        info_text = f"""
        Fecha de Generación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
        
        Este documento contiene:
        • Análisis de métricas de rendimiento
        • Matriz de correlaciones
        • Composición de la cartera
        • Recomendaciones de optimización
        """
        
        ax.text(0.5, 0.4, info_text, ha='center', fontsize=12,
               transform=ax.transAxes, verticalalignment='center',
               bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
        
        # Pie de página
        ax.text(0.5, 0.05, 'Sistema de Análisis de Carteras', 
               ha='center', fontsize=10, style='italic', transform=ax.transAxes)
        
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
    
    def _pdf_tabla_contenidos(self, pdf: PdfPages) -> None:
        """Crea página de tabla de contenidos."""
        fig = plt.figure(figsize=(8.5, 11))
        ax = fig.add_subplot(111)
        ax.axis('off')
        
        ax.text(0.5, 0.95, 'TABLA DE CONTENIDOS', 
               ha='center', fontsize=20, fontweight='bold', transform=ax.transAxes)
        
        contenidos = [
            "1. Portada",
            "2. Tabla de Contenidos",
            "3. Análisis de Activos Individuales",
            "4. Matriz de Correlaciones",
            "5. Composición de Cartera",
            "6. Recomendaciones",
            "7. Conclusiones"
        ]
        
        y_pos = 0.85
        for item in contenidos:
            ax.text(0.1, y_pos, item, fontsize=12, transform=ax.transAxes)
            y_pos -= 0.08
        
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
    
    def _pdf_analisis_tickers(self, pdf: PdfPages, metricas_dict: Dict) -> None:
        """Añade sección de análisis de tickers al PDF."""
        for ticker, metricas in metricas_dict.items():
            fig = plt.figure(figsize=(8.5, 11))
            ax = fig.add_subplot(111)
            ax.axis('off')
            
            # Título
            ax.text(0.5, 0.95, f'ANÁLISIS - {ticker}', 
                   ha='center', fontsize=18, fontweight='bold', transform=ax.transAxes)
            
            # Crear tabla de métricas
            metricas_display = []
            for clave, valor in metricas.items():
                if isinstance(valor, float):
                    if 'ratio' in str(clave).lower() or 'sharpe' in str(clave).lower():
                        metricas_display.append((str(clave), f"{valor:.4f}"))
                    else:
                        metricas_display.append((str(clave), f"{valor:.2%}"))
                else:
                    metricas_display.append((str(clave), str(valor)[:50]))
            
            # Tabla
            y_pos = 0.85
            for clave, valor in metricas_display[:15]:  # Primeras 15 métricas
                ax.text(0.1, y_pos, f"{clave}:", fontsize=10, fontweight='bold', 
                       transform=ax.transAxes)
                ax.text(0.6, y_pos, str(valor), fontsize=10, transform=ax.transAxes)
                y_pos -= 0.04
            
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
    
    def _pdf_seccion_correlaciones(self, pdf: PdfPages, correlacion_df: pd.DataFrame) -> None:
        """Añade sección de correlaciones al PDF."""
        # Heatmap
        fig = plt.figure(figsize=(10, 8))
        sns.heatmap(correlacion_df, annot=True, fmt='.2f',
                   cmap='RdYlGn', center=0, vmin=-1, vmax=1,
                   square=True, cbar_kws={'label': 'Correlación'})
        plt.title('Matriz de Correlación', fontsize=14, fontweight='bold')
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
    
    def _pdf_seccion_cartera(self, pdf: PdfPages, cartera_pesos: Dict) -> None:
        """Añade sección de composición de cartera al PDF."""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        
        tickers = list(cartera_pesos.keys())
        valores = list(cartera_pesos.values())
        colores = plt.cm.Set3(np.linspace(0, 1, len(tickers)))
        
        # Pastel
        ax1.pie(valores, labels=tickers, autopct='%1.1f%%', colors=colores, startangle=90)
        ax1.set_title('Composición de Cartera', fontsize=12, fontweight='bold')
        
        # Barras
        ax2.barh(tickers, valores, color=colores, edgecolor='black', linewidth=1.5)
        ax2.set_xlabel('Peso')
        ax2.set_title('Pesos por Activo', fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
    
    # ============================================
    # UTILIDADES
    # ============================================
    
    def _formatear_worksheet_excel(self, worksheet) -> None:
        """Aplica estilos básicos a un worksheet de Excel."""
        from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
        
        # Encabezado
        header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
        header_font = Font(bold=True, color="FFFFFF", size=11)
        
        thin_border = Border(
            left=Side(style='thin'),
            right=Side(style='thin'),
            top=Side(style='thin'),
            bottom=Side(style='thin')
        )
        
        # Aplicar a encabezado
        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.border = thin_border
        
        # Aplicar bordes a datos
        for row in worksheet.iter_rows(min_row=2, max_row=worksheet.max_row):
            for cell in row:
                cell.border = thin_border
                cell.alignment = Alignment(horizontal="center", vertical="center")
        
        # Ancho automático
        for column in worksheet.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            adjusted_width = min(max_length + 2, 50)
            worksheet.column_dimensions[column_letter].width = adjusted_width
    
    def exportar_resumen_texto(self, resumen_dict: Dict,
                              nombre_archivo: str = "Resumen_Analisis.txt") -> Path:
        """
        Exporta un resumen en formato texto.
        
        Args:
            resumen_dict: Diccionario con información a exportar
            nombre_archivo: Nombre del archivo
        
        Returns:
            Path del archivo
        """
        ruta_archivo = self.ruta_exportaciones / nombre_archivo
        
        with open(ruta_archivo, 'w', encoding='utf-8') as f:
            f.write("="*70 + "\n")
            f.write("RESUMEN DE ANÁLISIS DE CARTERA\n")
            f.write("="*70 + "\n")
            f.write(f"Fecha de generación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("="*70 + "\n\n")
            
            for seccion, contenido in resumen_dict.items():
                f.write(f"\n{seccion.upper()}\n")
                f.write("-" * 70 + "\n")
                if isinstance(contenido, dict):
                    for clave, valor in contenido.items():
                        f.write(f"  {clave}: {valor}\n")
                elif isinstance(contenido, list):
                    for item in contenido:
                        f.write(f"  • {item}\n")
                else:
                    f.write(f"{contenido}\n")
                f.write("\n")
        
        print(f"✓ Resumen exportado: {ruta_archivo}")
        return ruta_archivo
    
    def listar_exportaciones(self) -> List[Path]:
        """
        Lista todos los archivos exportados.
        
        Returns:
            Lista de rutas de archivos
        """
        archivos = sorted(list(self.ruta_exportaciones.glob("*")))
        
        if not archivos:
            print("❌ No hay archivos exportados")
            return []
        
        print(f"\n📁 Exportaciones ({len(archivos)} archivos):")
        for archivo in archivos:
            tamaño = archivo.stat().st_size / 1024  # KB
            print(f"   • {archivo.name} ({tamaño:.1f} KB)")
        
        return archivos
    
    def __repr__(self) -> str:
        return f"Exportador(ruta='{self.ruta_exportaciones}')"

## Tipos de Archivos Generados:

| Archivo | Descripción                               | Uso                     |
|---------|--------------------------------------------|--------------------------|
| **.xlsx** | Hojas de cálculo con datos y métricas      | Análisis detallado       |
| **.png**  | Imágenes de alta resolución (300 dpi)      | Presentaciones, informes |
| **.pdf**  | Reporte profesional completo               | Distribución, archivo    |
| **.txt**  | Resumen en texto plano                     | Referencia rápida        |


## main.py
Menú principal, orquestación de las clases. Unas 100 líneas.

In [11]:
"""
MAIN - Orquestador Principal de Análisis de Cartera
Coordina todas las clases para ejecutar análisis completo 
"""  

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Importar todas las clases
#from GestorRutas import GestorRutas
#from GestorCartera import GestorCartera
#from DescargadorYahoo import DescargadorYahoo
#from AnalizadorMetricas import AnalizadorMetricas
#from AnalizadorTecnico import AnalizadorTecnico
#from AnalizadorCartera import AnalizadorCartera
#from Exportador import Exportador

class MainAnalisisCartera:
    """
    Clase principal que orquesta todas las clases del sistema.
    Gestiona el flujo completo de análisis de cartera.
    """
    
    def __init__(self):
        """Inicializa todos los componentes del sistema."""
        print("\n" + "="*70)
        print("INICIALIZANDO SISTEMA DE ANÁLISIS DE CARTERA")
        print("="*70 + "\n")
        
        # Inicializar componentes
        self.gestor_rutas = GestorRutas(ruta_base="/home/enri/Py_Renta_4_2026")
        self.gestor_cartera = GestorCartera(self.gestor_rutas)
        self.descargador = DescargadorYahoo(self.gestor_rutas, usar_cache=True)
        self.analizador_metricas = AnalizadorMetricas(tasa_libre_riesgo=0.02)
        self.analizador_tecnico = AnalizadorTecnico()
        self.analizador_cartera = AnalizadorCartera(tasa_libre_riesgo=0.02)
        self.exportador = Exportador(self.gestor_rutas)
        
        self.datos_multiples = {}
        self.retornos = None
        self.cartera_actual = None
        self.metricas_activos = {}
        
        print("\n✓ Sistema inicializado correctamente\n")
    
    # ============================================
    # MENÚ PRINCIPAL
    # ============================================
    
    def menu_principal(self):
        """Muestra el menú principal."""
        while True:
            print("\n" + "="*70)
            print("MENÚ PRINCIPAL - ANÁLISIS DE CARTERA")
            print("="*70)
            print("""
1. Gestión de Cartera
2. Descargar Datos
3. Análisis Fundamental (Métricas)
4. Análisis Técnico
5. Análisis de Cartera
6. Exportar Resultados
7. Ver Información del Sistema
0. Salir
            """)
            
            opcion = input("Selecciona una opción: ").strip()
            
            if opcion == "1":
                self.submenu_cartera()
            elif opcion == "2":
                self.submenu_descargar()
            elif opcion == "3":
                self.submenu_metricas()
            elif opcion == "4":
                self.submenu_tecnico()
            elif opcion == "5":
                self.submenu_cartera_analisis()
            elif opcion == "6":
                self.submenu_exportar()
            elif opcion == "7":
                self.mostrar_info_sistema()
            elif opcion == "0":
                print("\n✓ Saliendo del programa...")
                break
            else:
                print("\n❌ Opción no válida")
    
    # ============================================
    # SUBMENU: GESTIÓN DE CARTERA
    # ============================================
    
    def submenu_cartera(self):
        """Submenu para gestionar carteras."""
        while True:
            print("\n" + "-"*70)
            print("GESTIÓN DE CARTERA")
            print("-"*70)
            print("""
1. Ver carteras existentes
2. Crear cartera nueva
3. Cargar cartera existente
4. Actualizar precios
5. Guardar cartera
6. Ver resumen de cartera
0. Volver al menú principal
            """)
            
            opcion = input("Selecciona una opción: ").strip()
            
            if opcion == "1":
                self.listar_carteras()
            elif opcion == "2":
                self.crear_cartera()
            elif opcion == "3":
                self.cargar_cartera()
            elif opcion == "4":
                self.actualizar_precios_cartera()
            elif opcion == "5":
                self.guardar_cartera()
            elif opcion == "6":
                self.resumen_cartera()
            elif opcion == "0":
                break
            else:
                print("\n❌ Opción no válida")
    
    def listar_carteras(self):
        """Lista todas las carteras disponibles y permite seleccionar una."""
        print("\n" + "="*70)
        carteras = self.gestor_cartera.listar_carteras_existentes()
        if not carteras:
            print("⚠️  No hay carteras disponibles")
            return
    
        # Pedir al usuario que seleccione una cartera
        indice = input("\nSelecciona el número de cartera (0 para cancelar): ").strip()
    
        try:
            indice = int(indice)
            if indice == 0:
                return  # Cancelar
            elif 1 <= indice <= len(carteras):
                cartera_nombre = carteras[indice - 1]
                self.gestor_cartera.cargar_cartera(cartera_nombre)
                self.cartera_actual = cartera_nombre
                print(f"\n✓ Cartera cargada: {cartera_nombre}")
            else:
                print("❌ Índice fuera de rango")
        except ValueError:
            print("❌ Error: ingresa un número válido")    
    def crear_cartera(self):
        """Crea una nueva cartera."""
        print("\n" + "="*70)
        nombre = input("Nombre de la cartera: ").strip()
        
        tickers_input = input("Tickers (separados por coma, ej: AAPL,MSFT,GOOGL): ").strip()
        tickers = [t.strip().upper() for t in tickers_input.split(",")]
        
        pesos_input = input("Pesos (separados por coma, ej: 0.3,0.4,0.3) [Enter para equiponderado]: ").strip()
        
        if pesos_input:
            try:
                pesos = [float(p.strip()) for p in pesos_input.split(",")]
            except ValueError:
                print("❌ Error: pesos deben ser números")
                return
        else:
            pesos = None
        
        try:
            self.gestor_cartera.crear_cartera_nueva(nombre, tickers, pesos)
            print(f"\n✓ Cartera '{nombre}' creada exitosamente")
        except Exception as e:
            print(f"\n❌ Error: {e}")
    
    def cargar_cartera(self):
        """Carga una cartera existente."""
        print("\n" + "="*70)
        carteras = self.gestor_cartera.listar_carteras_existentes()
        
        if not carteras:
            return
        
        indice = input("\nSelecciona el número de cartera: ").strip()
        
        try:
            indice = int(indice) - 1
            if 0 <= indice < len(carteras):
                cartera_nombre = carteras[indice]
                self.gestor_cartera.cargar_cartera(cartera_nombre)
                self.cartera_actual = cartera_nombre
                print(f"\n✓ Cartera cargada: {cartera_nombre}")
            else:
                print("❌ Índice fuera de rango")
        except ValueError:
            print("❌ Error: ingresa un número válido")
    
    def actualizar_precios_cartera(self):
        """Actualiza precios de la cartera cargada."""
        print("\n" + "="*70)
        
        if self.gestor_cartera.datos_cartera is None:
            print("❌ No hay cartera cargada")
            return
        
        tickers = self.gestor_cartera.obtener_tickers()
        print(f"\nObteniendo precios actuales para {len(tickers)} tickers...")
        
        try:
            precios = self.descargador.obtener_precios_actuales(tickers)
            self.gestor_cartera.actualizar_precios(precios)
            
            print("\n" + self.gestor_cartera.exportar_resumen_texto())
        except Exception as e:
            print(f"❌ Error: {e}")
    
    def guardar_cartera(self):
        """Guarda la cartera actual."""
        print("\n" + "="*70)
        
        if self.gestor_cartera.datos_cartera is None:
            print("❌ No hay cartera cargada para guardar")
            return
        
        try:
            self.gestor_cartera.guardar_cartera()
            print("\n✓ Cartera guardada exitosamente")
        except Exception as e:
            print(f"❌ Error: {e}")
    
    def resumen_cartera(self):
        """Muestra resumen de la cartera cargada."""
        print("\n" + "="*70)
        
        if self.gestor_cartera.datos_cartera is None:
            print("❌ No hay cartera cargada")
            return
        
        print(self.gestor_cartera.exportar_resumen_texto())
    
    # ============================================
    # SUBMENU: DESCARGAR DATOS
    # ============================================
    
    def submenu_descargar(self):
        """Submenu para descargar datos."""
        while True:
            print("\n" + "-"*70)
            print("DESCARGAR DATOS")
            print("-"*70)
            print("""
1. Descargar ticker individual
2. Descargar cartera completa
3. Obtener precio actual
4. Listar CSVs descargados
5. Leer CSV desde disco
0. Volver al menú principal
            """)
            
            opcion = input("Selecciona una opción: ").strip()
            
            if opcion == "1":
                self.descargar_ticker()
            elif opcion == "2":
                self.descargar_cartera_datos()
            elif opcion == "3":
                self.obtener_precio_actual()
            elif opcion == "4":
                self.listar_csvs()
            elif opcion == "5":
                self.leer_csv()
            elif opcion == "0":
                break
            else:
                print("\n❌ Opción no válida")
    
    def descargar_ticker(self):
        """Descarga datos de un ticker individual."""
        print("\n" + "="*70)
        ticker = input("Ticker a descargar (ej: AAPL): ").strip().upper()
        periodo = input("Período (1d/5d/1mo/3mo/6mo/1y/2y/5y/10y/max) [default: 1y]: ").strip() or "1y"
        
        try:
            print(f"\nDescargando {ticker}...")
            datos = self.descargador.descargar_datos(ticker, periodo=periodo)
            self.descargador.guardar_csv(ticker, datos)
            self.datos_multiples[ticker] = datos
            print(f"\n✓ {ticker} descargado: {len(datos)} registros")
        except Exception as e:
            print(f"\n❌ Error: {e}")
    
    def descargar_cartera_datos(self):
        """Descarga datos de todos los tickers de la cartera."""
        print("\n" + "="*70)
        
        if self.gestor_cartera.datos_cartera is None:
            print("❌ No hay cartera cargada")
            return
        
        tickers = self.gestor_cartera.obtener_tickers()
        periodo = input("Período (1y/5y/10y/max) [default: 1y]: ").strip() or "1y"
        
        print(f"\nDescargando {len(tickers)} tickers...")
        datos_dict = self.descargador.descargar_multiples(tickers, periodo=periodo)
        self.descargador.guardar_multiples_csv(datos_dict)
        self.datos_multiples.update(datos_dict)
        
        print(f"\n✓ {len(self.datos_multiples)} tickers en memoria")
    
    def obtener_precio_actual(self):
        """Obtiene el precio actual de un ticker."""
        print("\n" + "="*70)
        ticker = input("Ticker (ej: AAPL): ").strip().upper()
        
        try:
            precio = self.descargador.obtener_precio_actual(ticker)
            print(f"\n✓ Precio actual de {ticker}: ${precio:.2f}")
        except Exception as e:
            print(f"\n❌ Error: {e}")
    
    def listar_csvs(self):
        """Lista CSVs en la carpeta Datos_csv."""
        print("\n" + "="*70)
        csvs = self.descargador.listar_csvs()
    
    def leer_csv(self):
        """Lee un CSV desde disco."""
        print("\n" + "="*70)
        ticker = input("Ticker del CSV a leer (ej: AAPL): ").strip().upper()
        
        try:
            datos = self.descargador.leer_csv(ticker)
            self.datos_multiples[ticker] = datos
            print(f"\n✓ {ticker} leído: {len(datos)} registros")
        except Exception as e:
            print(f"\n❌ Error: {e}")
    
    # ============================================
    # SUBMENU: ANÁLISIS FUNDAMENTAL
    # ============================================
    
    def submenu_metricas(self):
        """Submenu para análisis fundamental."""
        while True:
            print("\n" + "-"*70)
            print("ANÁLISIS FUNDAMENTAL (MÉTRICAS)")
            print("-"*70)
            print("""
1. Calcular métricas de un ticker
2. Calcular métricas de múltiples tickers
3. Ver diagnóstico de activo
4. Comparar múltiples activos
0. Volver al menú principal
            """)
            
            opcion = input("Selecciona una opción: ").strip()
            
            if opcion == "1":
                self.metricas_ticker()
            elif opcion == "2":
                self.metricas_multiples()
            elif opcion == "3":
                self.diagnostico_activo()
            elif opcion == "4":
                self.comparar_activos()
            elif opcion == "0":
                break
            else:
                print("\n❌ Opción no válida")
    
    def metricas_ticker(self):
        """Calcula métricas para un ticker."""
        print("\n" + "="*70)
        
        if not self.datos_multiples:
            print("❌ No hay datos cargados")
            return
        
        print("Tickers disponibles:")
        tickers = list(self.datos_multiples.keys())
        for i, ticker in enumerate(tickers, 1):
            print(f"  {i}. {ticker}")
        
        indice = input("\nSelecciona ticker: ").strip()
        
        try:
            indice = int(indice) - 1
            if 0 <= indice < len(tickers):
                ticker = tickers[indice]
                datos = self.datos_multiples[ticker]
                
                print(f"\nCalculando métricas para {ticker}...")
                metricas = self.analizador_metricas.calcular_metricas_csv(datos, nombre_activo=ticker)
                self.metricas_activos[ticker] = metricas
                
                print(self.analizador_metricas.exportar_resumen_metricas(metricas))
        except (ValueError, IndexError):
            print("❌ Selección inválida")
    
    def metricas_multiples(self):
        """Calcula métricas para múltiples tickers."""
        print("\n" + "="*70)
        
        if not self.datos_multiples:
            print("❌ No hay datos cargados")
            return
        
        print(f"Calculando métricas para {len(self.datos_multiples)} tickers...")
        
        for ticker, datos in self.datos_multiples.items():
            try:
                metricas = self.analizador_metricas.calcular_metricas_csv(datos, nombre_activo=ticker)
                self.metricas_activos[ticker] = metricas
            except Exception as e:
                print(f"⚠️  {ticker}: {str(e)[:50]}")
        
        print(f"\n✓ Métricas calculadas para {len(self.metricas_activos)} tickers")
    
    def diagnostico_activo(self):
        """Muestra diagnóstico de un activo."""
        print("\n" + "="*70)
        
        if not self.metricas_activos:
            print("❌ No hay métricas calculadas")
            return
        
        tickers = list(self.metricas_activos.keys())
        print("Activos disponibles:")
        for i, ticker in enumerate(tickers, 1):
            print(f"  {i}. {ticker}")
        
        indice = input("\nSelecciona activo: ").strip()
        
        try:
            indice = int(indice) - 1
            if 0 <= indice < len(tickers):
                ticker = tickers[indice]
                metricas = self.metricas_activos[ticker]
                diagnostico = self.analizador_metricas.calcular_diagnostico(metricas)
                
                print(f"\nDiagnóstico de {ticker}:")
                print(f"Puntuación: {diagnostico['puntuacion']}/10")
                print("\nEvaluaciones:")
                for evaluacion in diagnostico['evaluaciones']:
                    print(f"  {evaluacion}")
        except (ValueError, IndexError):
            print("❌ Selección inválida")
    
    def comparar_activos(self):
        """Compara múltiples activos."""
        print("\n" + "="*70)
        
        if len(self.metricas_activos) < 2:
            print("❌ Se necesitan al menos 2 activos para comparar")
            return
        
        # Crear DataFrame de comparación
        comparativa_data = []
        
        for ticker, metricas in self.metricas_activos.items():
            comparativa_data.append({
                'Ticker': ticker,
                'Retorno': f"{metricas.get('retorno_anualizado', 0):.2%}",
                'Volatilidad': f"{metricas.get('volatilidad_anualizada', 0):.2%}",
                'Sharpe': f"{metricas.get('sharpe_ratio', 0):.4f}",
                'Sortino': f"{metricas.get('sortino_ratio', 0):.4f}",
                'Drawdown': f"{metricas.get('drawdown_maximo', 0):.2%}"
            })
        
        df_comparativa = pd.DataFrame(comparativa_data)
        print("\n" + df_comparativa.to_string(index=False))
    
    # ============================================
    # SUBMENU: ANÁLISIS TÉCNICO
    # ============================================
    
    def submenu_tecnico(self):
        """Submenu para análisis técnico."""
        while True:
            print("\n" + "-"*70)
            print("ANÁLISIS TÉCNICO")
            print("-"*70)
            print("""
1. Análisis técnico de un ticker
2. Diagnóstico técnico
3. Generar gráficas técnicas
4. Análisis de múltiples tickers
0. Volver al menú principal
            """)
            
            opcion = input("Selecciona una opción: ").strip()
            
            if opcion == "1":
                self.tecnico_ticker()
            elif opcion == "2":
                self.diagnostico_tecnico()
            elif opcion == "3":
                self.graficas_tecnicas()
            elif opcion == "4":
                self.tecnico_multiples()
            elif opcion == "0":
                break
            else:
                print("\n❌ Opción no válida")
    
    def tecnico_ticker(self):
        """Análisis técnico de un ticker."""
        print("\n" + "="*70)
        
        if not self.datos_multiples:
            print("❌ No hay datos cargados")
            return
        
        tickers = list(self.datos_multiples.keys())
        print("Tickers disponibles:")
        for i, ticker in enumerate(tickers, 1):
            print(f"  {i}. {ticker}")
        
        indice = input("\nSelecciona ticker: ").strip()
        
        try:
            indice = int(indice) - 1
            ticker = tickers[indice]
            datos = self.datos_multiples[ticker]
            
            # Calcular indicadores
            print(f"\nIndicadores técnicos para {ticker}:")
            
            rsi = self.analizador_tecnico.calcular_rsi(datos).iloc[-1]
            macd_data = self.analizador_tecnico.calcular_macd(datos)
            atr = self.analizador_tecnico.calcular_atr(datos).iloc[-1]
            
            print(f"  RSI (14): {rsi:.2f}")
            print(f"  MACD: {macd_data['macd'].iloc[-1]:.4f}")
            print(f"  ATR (14): ${atr:.2f}")
        except (ValueError, IndexError):
            print("❌ Selección inválida")
    
    def diagnostico_tecnico(self):
        """Diagnóstico técnico."""
        print("\n" + "="*70)
        
        if not self.datos_multiples:
            print("❌ No hay datos cargados")
            return
        
        tickers = list(self.datos_multiples.keys())
        print("Tickers disponibles:")
        for i, ticker in enumerate(tickers, 1):
            print(f"  {i}. {ticker}")
        
        indice = input("\nSelecciona ticker: ").strip()
        
        try:
            indice = int(indice) - 1
            ticker = tickers[indice]
            datos = self.datos_multiples[ticker]
            
            diagnostico = self.analizador_tecnico.diagnostico_tecnico(datos, nombre_activo=ticker)
            print(self.analizador_tecnico.exportar_resumen_tecnico(diagnostico))
        except (ValueError, IndexError):
            print("❌ Selección inválida")
    
    def graficas_tecnicas(self):
        """Genera gráficas técnicas."""
        print("\n" + "="*70)
        
        if not self.datos_multiples:
            print("❌ No hay datos cargados")
            return
        
        tickers = list(self.datos_multiples.keys())
        print("Tickers disponibles:")
        for i, ticker in enumerate(tickers, 1):
            print(f"  {i}. {ticker}")
        
        indice = input("\nSelecciona ticker: ").strip()
        
        try:
            indice = int(indice) - 1
            ticker = tickers[indice]
            datos = self.datos_multiples[ticker]
            
            print(f"\nGenerando gráficas para {ticker}...")
            self.analizador_tecnico.grafica_completa(datos, nombre_activo=ticker)
        except (ValueError, IndexError):
            print("❌ Selección inválida")
    
    def tecnico_multiples(self):
        """Análisis técnico de múltiples tickers."""
        print("\n" + "="*70)
        
        if len(self.datos_multiples) < 2:
            print("❌ Se necesitan al menos 2 tickers")
            return
        
        tecnico_data = []
        
        for ticker, datos in self.datos_multiples.items():
            try:
                rsi = self.analizador_tecnico.calcular_rsi(datos).iloc[-1]
                macd = self.analizador_tecnico.calcular_macd(datos)
                
                tecnico_data.append({
                    'Ticker': ticker,
                    'RSI': f"{rsi:.2f}",
                    'MACD': 'Alcista' if macd['histogram'].iloc[-1] > 0 else 'Bajista',
                    'Tendencia': self.analizador_tecnico.diagnostico_tecnico(datos)['tendencia']
                })
            except:
                pass
        
        df_tecnico = pd.DataFrame(tecnico_data)
        print("\n" + df_tecnico.to_string(index=False))
    
    # ============================================
    # SUBMENU: ANÁLISIS DE CARTERA
    # ============================================
    
    def submenu_cartera_analisis(self):
        """Submenu para análisis de cartera."""
        while True:
            print("\n" + "-"*70)
            print("ANÁLISIS DE CARTERA")
            print("-"*70)
            print("""
1. Cartera de mínima varianza
2. Cartera de máximo Sharpe
3. Cartera de máximo retorno
4. Frontera eficiente
5. Matriz de correlación
6. Análisis de riesgo
7. Comparar carteras óptimas
0. Volver al menú principal
            """)
            
            opcion = input("Selecciona una opción: ").strip()
            
            if opcion == "1":
                self.cartera_min_var()
            elif opcion == "2":
                self.cartera_max_sharpe()
            elif opcion == "3":
                self.cartera_max_ret()
            elif opcion == "4":
                self.frontera_eficiente()
            elif opcion == "5":
                self.matriz_correlacion()
            elif opcion == "6":
                self.analisis_riesgo()
            elif opcion == "7":
                self.comparar_carteras()
            elif opcion == "0":
                break
            else:
                print("\n❌ Opción no válida")
    
    def cartera_min_var(self):
        """Calcula cartera de mínima varianza."""
        print("\n" + "="*70)
        
        if not self._preparar_retornos():
            return
        
        print("Calculando cartera de mínima varianza...")
        cartera = self.analizador_cartera.cartera_minima_varianza(self.retornos)
        
        self._mostrar_cartera(cartera, "Mínima Varianza")
    
    def cartera_max_sharpe(self):
        """Calcula cartera de máximo Sharpe."""
        print("\n" + "="*70)
        
        if not self._preparar_retornos():
            return
        
        print("Calculando cartera de máximo Sharpe...")
        cartera = self.analizador_cartera.cartera_maximo_sharpe(self.retornos)
        
        self._mostrar_cartera(cartera, "Máximo Sharpe")
    
    def cartera_max_ret(self):
        """Calcula cartera de máximo retorno."""
        print("\n" + "="*70)
        
        if not self._preparar_retornos():
            return
        
        print("Calculando cartera de máximo retorno...")
        cartera = self.analizador_cartera.cartera_maxima_rentabilidad(self.retornos)
        
        self._mostrar_cartera(cartera, "Máximo Retorno")
    
    def frontera_eficiente(self):
        """Muestra la frontera eficiente."""
        print("\n" + "="*70)
        
        if not self._preparar_retornos():
            return
        
        print("Generando frontera eficiente...")
        self.analizador_cartera.grafica_frontera_eficiente(self.retornos, num_carteras=1000)
    
    def matriz_correlacion(self):
        """Muestra matriz de correlación."""
        print("\n" + "="*70)
        
        if not self._preparar_retornos():
            return
        
        print("Calculando matriz de correlación...")
        correlacion = self.analizador_cartera.calcular_matriz_correlacion(self.retornos)
        
        print("\n" + str(correlacion.round(3)))
        
        print("\nGenerando heatmap...")
        self.analizador_cartera.grafica_correlacion(self.retornos)
    
    def analisis_riesgo(self):
        """Análisis de riesgo de la cartera."""
        print("\n" + "="*70)
        print(self.analizador_cartera.exportar_resumen_cartera())
    
    def comparar_carteras(self):
        """Compara carteras óptimas."""
        print("\n" + "="*70)
        
        if not self._preparar_retornos():
            return
        
        print(self.analizador_cartera.exportar_comparacion_carteras_optimas(self.retornos))
    
    # ============================================
    # SUBMENU: EXPORTAR
    # ============================================
    
    def submenu_exportar(self):
        """Submenu para exportar resultados."""
        while True:
            print("\n" + "-"*70)
            print("EXPORTAR RESULTADOS")
            print("-"*70)
            print("""
1. Exportar métricas a Excel
2. Exportar correlaciones a Excel
3. Generar heatmap de correlación
4. Generar gráfico subacuático
5. Generar PDF completo
6. Listar archivos exportados
0. Volver al menú principal
            """)
            
            opcion = input("Selecciona una opción: ").strip()
            
            if opcion == "1":
                self.exportar_metricas()
            elif opcion == "2":
                self.exportar_correlaciones()
            elif opcion == "3":
                self.exportar_heatmap()
            elif opcion == "4":
                self.exportar_subacuatico()
            elif opcion == "5":
                self.exportar_pdf()
            elif opcion == "6":
                self.listar_exportaciones()
            elif opcion == "0":
                break
            else:
                print("\n❌ Opción no válida")
    
    def exportar_metricas(self):
        """Exporta métricas a Excel."""
        print("\n" + "="*70)
        
        if not self.metricas_activos:
            print("❌ No hay métricas para exportar")
            return
        
        ruta = self.exportador.exportar_metricas_excel(self.metricas_activos)
        print(f"\n✓ Archivo: {ruta}")
    
    def exportar_correlaciones(self):
        """Exporta correlaciones a Excel."""
        print("\n" + "="*70)
        
        if not self._preparar_retornos():
            return
        
        correlacion = self.analizador_cartera.calcular_matriz_correlacion(self.retornos)
        ruta = self.exportador.exportar_correlaciones_excel(correlacion)
        print(f"\n✓ Archivo: {ruta}")
    
    def exportar_heatmap(self):
        """Exporta heatmap de correlación."""
        print("\n" + "="*70)
        
        if not self._preparar_retornos():
            return
        
        correlacion = self.analizador_cartera.calcular_matriz_correlacion(self.retornos)
        ruta = self.exportador.grafica_heatmap_correlacion(correlacion)
        print(f"\n✓ Archivo: {ruta}")
    
    def exportar_subacuatico(self):
        """Exporta gráfico subacuático."""
        print("\n" + "="*70)
        
        if not self.datos_multiples:
            print("❌ No hay datos cargados")
            return
        
        tickers = list(self.datos_multiples.keys())
        print("Tickers disponibles:")
        for i, ticker in enumerate(tickers, 1):
            print(f"  {i}. {ticker}")
        
        indice = input("\nSelecciona ticker: ").strip()
        
        try:
            indice = int(indice) - 1
            ticker = tickers[indice]
            datos = self.datos_multiples[ticker]
            
            ruta = self.exportador.grafica_grafico_subacuatico(datos)
            print(f"\n✓ Archivo: {ruta}")
        except (ValueError, IndexError):
            print("❌ Selección inválida")
    
    def exportar_pdf(self):
        """Exporta PDF completo."""
        print("\n" + "="*70)
        print("⚠️  Preparando datos...")
        
        if not self._preparar_retornos():
            return
        
        if not self.metricas_activos:
            print("⚠️  Calculando métricas...")
            self.metricas_multiples()
        
        correlacion = self.analizador_cartera.calcular_matriz_correlacion(self.retornos)
        
        # Cartera actual o máximo Sharpe
        if self.cartera_actual:
            cartera_pesos = dict(zip(
                self.gestor_cartera.obtener_tickers(),
                self.gestor_cartera.datos_cartera['Peso'].values
            ))
        else:
            cartera = self.analizador_cartera.cartera_maximo_sharpe(self.retornos)
            cartera_pesos = cartera['pesos']
        
        print("Generando PDF...")
        ruta = self.exportador.exportar_pdf_completo(
            datos_dict=self.datos_multiples,
            metricas_dict=self.metricas_activos,
            correlacion_df=correlacion,
            cartera_pesos=cartera_pesos
        )
        
        print(f"\n✓ Archivo: {ruta}")
    
    def listar_exportaciones(self):
        """Lista archivos exportados."""
        print("\n" + "="*70)
        self.exportador.listar_exportaciones()
    
    # ============================================
    # FUNCIONES AUXILIARES
    # ============================================
    
    def _preparar_retornos(self) -> bool:
        """Prepara retornos si no están listos."""
        if self.retornos is not None:
            return True
        
        if not self.datos_multiples:
            print("❌ No hay datos cargados")
            return False
        
        print("Calculando retornos...")
        self.retornos = self.analizador_cartera.calcular_retornos_diarios(self.datos_multiples)
        return True
    
    def _mostrar_cartera(self, cartera: Dict, nombre: str):
        """Muestra información de una cartera."""
        print(f"\n{'='*70}")
        print(f"CARTERA: {nombre}")
        print(f"{'='*70}")
        
        print(f"\nComposición:")
        for ticker, peso in cartera['pesos'].items():
            print(f"  {ticker:8} {peso:6.2%}")
        
        print(f"\nCaracterísticas:")
        print(f"  Retorno esperado: {cartera['retorno']:.2%}")
        print(f"  Volatilidad: {cartera['volatilidad']:.2%}")
        print(f"  Sharpe Ratio: {cartera['sharpe']:.4f}")
    
    def mostrar_info_sistema(self):
        """Muestra información del sistema."""
        print("\n" + "=" * 70)
        print("INFORMACIÓN DEL SISTEMA")
        print("=" * 70)
    
        info = self.gestor_rutas.obtener_info_entorno()
    
        print(f"\nEntorno: {info['entorno']}")
        print(f"Usuario: {info['usuario']}")
        print(f"Sistema operativo: {info['sistema_operativo']}")
        print(f"Directorio actual: {info['directorio_actual']}")
        print(f"Datos CSV accesibles: {'✓' if info['datos_csv_accesible'] else '✗'}")
    
        print(f"\nRutas configuradas:")
        for nombre, ruta in info['rutas'].items():
            print(f"  {nombre:20} -> {ruta}")
    
        print(f"\nDatos en memoria:")
        print(f"  Tickers cargados: {len(self.datos_multiples)}")
        print(f"  Métricas calculadas: {len(self.metricas_activos)}")
        print(f"  Cartera cargada: {'✓' if self.cartera_actual else '✗'}")
# ============================================
# PUNTO DE ENTRADA
# ============================================

if __name__ == "__main__":
    try:
        app = MainAnalisisCartera()
        app.menu_principal()
    except KeyboardInterrupt:
        print("\n\n✓ Programa interrumpido por el usuario")
    except Exception as e:
        print(f"\n❌ Error fatal: {e}")


INICIALIZANDO SISTEMA DE ANÁLISIS DE CARTERA

✓ Directorio: /home/enri/Py_Renta_4_2026
✓ Directorio: /home/enri/Py_Renta_4_2026/Exportaciones
✓ Directorio: /home/enri/Py_Renta_4_2026/Carteras
✓ Directorio: /home/enri/Py_Renta_4_2026/Logs
✓ Directorio: /home/enri/Py_Renta_4_2026

✓ Sistema inicializado correctamente


MENÚ PRINCIPAL - ANÁLISIS DE CARTERA

1. Gestión de Cartera
2. Descargar Datos
3. Análisis Fundamental (Métricas)
4. Análisis Técnico
5. Análisis de Cartera
6. Exportar Resultados
7. Ver Información del Sistema
0. Salir
            


Selecciona una opción:  1



----------------------------------------------------------------------
GESTIÓN DE CARTERA
----------------------------------------------------------------------

1. Ver carteras existentes
2. Crear cartera nueva
3. Cargar cartera existente
4. Actualizar precios
5. Guardar cartera
6. Ver resumen de cartera
0. Volver al menú principal
            


Selecciona una opción:  1



📁 Carteras encontradas: 4
   1. mi_cartera_2026_rev1.xlsx
   2. mi_cartera_2026_rev1_fiscal.xlsx
   3. mi_cartera_2026py_0.xlsx
   4. pruebas.xlsx


...


Calculando métricas para 6 tickers...
❌ Error calculando métricas: float division by zero

✓ Métricas calculadas para 6 tickers

----------------------------------------------------------------------
ANÁLISIS FUNDAMENTAL (MÉTRICAS)
----------------------------------------------------------------------

1. Calcular métricas de un ticker
2. Calcular métricas de múltiples tickers
3. Ver diagnóstico de activo
4. Comparar múltiples activos
0. Volver al menú principal
            


Selecciona una opción:  3



Activos disponibles:
  1. 0P00012NJH.F
  2. 0P00019MZZ.F
  3. 0P0000KY8J.F
  4. 0P0000YLAY.F
  5. 0P0001RCAQ.F
  6. DI4C.F



Selecciona activo:  1



Diagnóstico de 0P00012NJH.F:

❌ Error fatal: 'puntuacion'


## Como ejecutar